## Imports


In [1]:
import argparse
import functools
import gc
import hashlib
import json
import math
import numbers
import os
import random
import shutil
import sys
import urllib.request
import zipfile
from dataclasses import asdict, dataclass
from io import BytesIO
from pathlib import Path
from time import perf_counter
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Set, TextIO, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from numba import njit
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, Sampler

try:
    from flash_attn import flash_attn_varlen_qkvpacked_func
except ImportError:
    flash_attn_varlen_qkvpacked_func = None

try:
    from torch.nn.attention.flex_attention import create_block_mask, flex_attention
except ImportError:
    create_block_mask = None
    flex_attention = None

try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
except Exception:
    plt = None
    ListedColormap = None


import warnings
warnings.filterwarnings("ignore", message=".*bfloat16 compilation natively.*")
warnings.filterwarnings("ignore", message=".*Not enough SMs.*")
warnings.filterwarnings("ignore", message=".*epoch parameter in.*scheduler.*")

print("All imports loaded successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"flash_attn available: {flash_attn_varlen_qkvpacked_func is not None}")


All imports loaded successfully.
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
flash_attn available: False


## Dataset Setup


In [2]:
from pathlib import Path
import json

# Colab + Google Drive setup
IS_COLAB = False
try:
    from google.colab import drive
    IS_COLAB = True
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print(f"Not running in Colab or Drive mount unavailable: {exc}")

WORK_DIR = Path('/content/drive/MyDrive/DL_A2') if IS_COLAB else Path.cwd()
DATA_DIR = WORK_DIR / "data"
CKPT_DIR = WORK_DIR / "saved_models"

for p in [DATA_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CHALLENGES_FILE = DATA_DIR / "challenges.json"
SOLUTIONS_FILE = DATA_DIR / "solutions.json"

# Expected ARC location in Drive: DL_A2/ARC-AGI-master/data/{training,evaluation}
ARC_DATA_ROOT = WORK_DIR / "ARC-AGI-master" / "data"
TRAIN_DIR = ARC_DATA_ROOT / "training"
EVAL_DIR = ARC_DATA_ROOT / "evaluation"

if not TRAIN_DIR.exists() or not EVAL_DIR.exists():
    raise FileNotFoundError(
        f"Expected ARC folders at {ARC_DATA_ROOT}. Make sure Drive has "
        "DL_A2/ARC-AGI-master/data/training and .../evaluation."
    )

print(f"Using project root: {WORK_DIR}")
print(f"Found ARC-AGI data at: {ARC_DATA_ROOT}")
print(f"  Training dir : {TRAIN_DIR}  ({len(list(TRAIN_DIR.glob('*.json')))} tasks)")
print(f"  Evaluation dir: {EVAL_DIR}  ({len(list(EVAL_DIR.glob('*.json')))} tasks)")


def _read_task(path: Path) -> dict:
    """Load a single ARC task JSON and return its train/test splits."""
    d = json.loads(path.read_text())
    return {"train": d["train"], "test": d["test"]}


def _fold_test_into_train(tasks: dict) -> dict:
    """Merge test pairs into train so all labelled pairs are seen during training."""
    out = {}
    for tid, t in tasks.items():
        merged = dict(t)
        merged["train"] = t.get("train", []) + t.get("test", [])
        merged.pop("test", None)
        out[tid] = merged
    return out


def build_dataset(train_dir: Path, eval_dir: Path):
    """Read all ARC tasks and write challenges.json / solutions.json."""
    raw_train: dict = {}
    eval_challenges: dict = {}
    eval_solutions: dict = {}

    for fp in sorted(train_dir.glob("*.json")):
        raw_train[fp.stem] = _read_task(fp)

    for fp in sorted(eval_dir.glob("*.json")):
        t = _read_task(fp)
        eval_challenges[fp.stem] = {
            "train": t["train"],
            "test": [{"input": x["input"]} for x in t["test"]],
        }
        eval_solutions[fp.stem] = [x["output"] for x in t["test"]]

    all_challenges = {**_fold_test_into_train(raw_train), **eval_challenges}
    CHALLENGES_FILE.write_text(json.dumps(all_challenges))
    SOLUTIONS_FILE.write_text(json.dumps(eval_solutions))
    return len(raw_train), len(eval_challenges)


if CHALLENGES_FILE.exists() and SOLUTIONS_FILE.exists():
    n = len(json.loads(CHALLENGES_FILE.read_text()))
    print(f"Dataset already built - {n} tasks in challenges.json")
else:
    n_tr, n_ev = build_dataset(TRAIN_DIR, EVAL_DIR)
    print(f"Built: {n_tr} train + {n_ev} eval = {n_tr + n_ev} total tasks")
    print(f"  challenges.json -> {CHALLENGES_FILE}")
    print(f"  solutions.json  -> {SOLUTIONS_FILE}")

Mounted at /content/drive
Using project root: /content/drive/MyDrive/DL_A2
Found ARC-AGI data at: /content/drive/MyDrive/DL_A2/ARC-AGI-master/data
  Training dir : /content/drive/MyDrive/DL_A2/ARC-AGI-master/data/training  (400 tasks)
  Evaluation dir: /content/drive/MyDrive/DL_A2/ARC-AGI-master/data/evaluation  (400 tasks)
Dataset already built - 800 tasks in challenges.json


## Tokenisation & Augmentation


In [3]:
N_VOCAB = 14

SPECIAL_TOKS = ["<start>", "<next_line>", "<input_output_separator>", "<end>"]

TOK2ID: Dict[str, int] = {str(i): i for i in range(10)}
for _offset, _token in enumerate(SPECIAL_TOKS, start=10):
    TOK2ID[_token] = _offset

ID2TOK: Dict[int, str] = {idx: token for token, idx in TOK2ID.items()}

START_ID = TOK2ID["<start>"]
NL_ID    = TOK2ID["<next_line>"]
SEP_ID   = TOK2ID["<input_output_separator>"]
END_ID   = TOK2ID["<end>"]

MAX_LEN   = 512
PAD_LABEL = -100


# Color extraction helpers

def _extract_task_colors(task: Dict[str, object], key: str) -> Set[int]:
    colors: Set[int] = set()
    for split in ("train", "test"):
        for pair in task.get(split, []):
            for row in pair.get(key) or []:
                for val in row:
                    val_i = int(val)
                    if 1 <= val_i <= 9:
                        colors.add(val_i)
    return colors


def get_input_colors(task: Dict[str, object]) -> List[int]:
    input_colors  = _extract_task_colors(task, "input")
    output_colors = _extract_task_colors(task, "output")
    output_only   = output_colors - input_colors
    return [c for c in range(1, 10) if c not in output_only]


# Dihedral (D4) grid transforms

_DIHEDRAL_NAMES = [
    "identity",
    "rot90",
    "rot180",
    "rot270",
    "flip_horizontal",
    "flip_vertical",
    "flip_main_diagonal",
    "flip_anti_diagonal",
]

_DIHEDRAL_INVERSES: Dict[str, str] = {
    "identity":           "identity",
    "rot90":              "rot270",
    "rot180":             "rot180",
    "rot270":             "rot90",
    "flip_horizontal":    "flip_horizontal",
    "flip_vertical":      "flip_vertical",
    "flip_main_diagonal": "flip_main_diagonal",
    "flip_anti_diagonal": "flip_anti_diagonal",
}


def _copy(grid):              return [list(row) for row in grid]
def _rot90(grid):             return [list(row) for row in zip(*grid[::-1])] if grid else []
def _rot180(grid):            return [list(reversed(row)) for row in reversed(grid)]
def _rot270(grid):            return [list(row) for row in zip(*grid)][::-1] if grid else []
def _flip_h(grid):            return [list(reversed(row)) for row in grid]
def _flip_v(grid):            return [list(row) for row in reversed(grid)]
def _flip_main_diag(grid):    return [list(row) for row in zip(*grid)] if grid else []
def _flip_anti_diag(grid):    return _flip_v(_rot90(grid))

_DIHEDRAL_FNS: Dict[str, Callable] = {
    "identity":           _copy,
    "rot90":              _rot90,
    "rot180":             _rot180,
    "rot270":             _rot270,
    "flip_horizontal":    _flip_h,
    "flip_vertical":      _flip_v,
    "flip_main_diagonal": _flip_main_diag,
    "flip_anti_diagonal": _flip_anti_diag,
}


def _validate_transform_index(transform_index: int) -> str:
    if transform_index < 0:
        raise ValueError("transform_index must be non-negative.")
    return _DIHEDRAL_NAMES[transform_index % 8]


def rotate_grid(grid: Sequence[Sequence[int]], transform_index: int) -> List[List[int]]:
    name = _validate_transform_index(transform_index)
    return _DIHEDRAL_FNS[name](grid)


def rotate_grid_inverse(grid: Sequence[Sequence[int]], transform_index: int) -> List[List[int]]:
    name = _validate_transform_index(transform_index)
    return _DIHEDRAL_FNS[_DIHEDRAL_INVERSES[name]](grid)


# Color permutation helpers

def permute_token_colors(tokens: Sequence[int], mapping: Sequence[int]) -> List[int]:
    return [int(mapping[tok] if 0 <= tok < len(mapping) else tok) for tok in tokens]


def permute_grid_colors(grid: Sequence[Sequence[int]], mapping: Sequence[int]) -> List[List[int]]:
    return [
        [int(mapping[val] if 0 <= val < len(mapping) else val) for val in row]
        for row in grid
    ]


# Tokenisation

def _cell_to_token(value: int) -> int:
    if value not in range(10):
        raise ValueError(f"Grid values must be digits in [0, 9], received {value}")
    return value


def grid_to_tok(grid: Iterable[Iterable[int]]) -> List[int]:
    tokens: List[int] = []
    for row in grid:
        for value in row:
            tokens.append(_cell_to_token(int(value)))
        tokens.append(NL_ID)
    return tokens


def tokenize_pair(
    input_grid: Iterable[Iterable[int]],
    output_grid: Optional[Iterable[Iterable[int]]] = None,
    include_output: bool = True,
    append_end: bool = True,
) -> List[int]:
    tokens = [START_ID] + grid_to_tok(input_grid) + [SEP_ID]
    if include_output and output_grid is not None:
        tokens.extend(grid_to_tok(output_grid))
    if append_end:
        tokens.append(END_ID)
    return tokens


def load_tasks(json_path: Path) -> Dict[str, dict]:
    with Path(json_path).open("r") as f:
        return json.load(f)


# Detokenisation

def tok_to_grid(tokens: Sequence[int]) -> List[List[int]]:
    rows: List[List[int]] = []
    current_row: List[int] = []
    for token in tokens:
        if token == NL_ID:
            if current_row:
                rows.append(current_row)
            current_row = []
            continue
        if 0 <= token <= 9:
            current_row.append(token)
        else:
            break
    if current_row:
        rows.append(current_row)
    return rows


def get_output_tokens(sequence: Sequence[int]) -> List[int]:
    past_sep = False
    outputs: List[int] = []
    for token in sequence:
        if not past_sep:
            if token == SEP_ID:
                past_sep = True
            continue
        if token == END_ID:
            break
        outputs.append(token)
    return outputs


def tok_to_str(tokens: Sequence[int]) -> str:
    return " ".join(ID2TOK.get(int(tok), str(int(tok))) for tok in tokens)


def split_grids(tokens: Sequence[int]) -> List[List[List[int]]]:
    grids: List[List[List[int]]] = []
    current_grid: List[List[int]] = []
    current_row: List[int] = []

    for tok in tokens:
        if tok == END_ID:
            break
        if tok == START_ID:
            continue
        if tok == SEP_ID:
            if current_row:
                current_grid.append(current_row)
                current_row = []
            if current_grid:
                grids.append(current_grid)
            current_grid = []
            continue
        if tok == NL_ID:
            if current_row:
                current_grid.append(current_row)
            current_row = []
            continue
        if 0 <= tok <= 9:
            current_row.append(int(tok))
        else:
            break

    if current_row:
        current_grid.append(current_row)
    if current_grid:
        grids.append(current_grid)
    return grids


def is_valid_grid(grid: Sequence[Sequence[int]]) -> bool:
    if not grid or not grid[0]:
        return False
    row_len = len(grid[0])
    return all(len(row) == row_len for row in grid)


# Numba-accelerated 3D position computation

@njit
def _fill_3d_positions_numba(ids, mask, out, start_id, sep_id, end_id, nl_id):
    B, S = ids.shape
    for b in range(B):
        x = y = 0
        z = 1
        for t in range(S):
            if not mask[b, t]:
                continue
            val = ids[b, t]

            if val == start_id:
                out[b, t] = (0, 0, 0)
                x = y = 0; z = 1
                continue

            if val == sep_id:
                out[b, t] = (0, 0, 2)
                x = y = 0; z = 3
                continue

            if val == end_id:
                out[b, t] = (0, 0, 4)
                continue

            out[b, t, 0] = max(0, min(x, 30))
            out[b, t, 1] = max(0, min(y, 29))
            out[b, t, 2] = z

            if val == nl_id:
                x = 0; y += 1
            else:
                x += 1


def get_3d_positions(
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    if input_ids.dim() != 2:
        raise ValueError("input_ids must have shape [batch, seq_len].")

    ids_np  = input_ids.detach().cpu().numpy()
    mask_np = (
        attention_mask.detach().cpu().numpy().astype(bool)
        if attention_mask is not None
        else np.ones_like(ids_np, dtype=bool)
    )

    B, S  = ids_np.shape
    pos_np = np.zeros((B, S, 3), dtype=np.int64)
    _fill_3d_positions_numba(ids_np, mask_np, pos_np, START_ID, SEP_ID, END_ID, NL_ID)
    return torch.from_numpy(pos_np).to(device=input_ids.device)


# Dataset types

@dataclass
class PuzzleSample:
    tokens:                        torch.LongTensor
    sep_index:                     int
    example_id:                    int
    task_id:                       str
    split:                         str
    pair_index:                    int
    has_output:                    bool
    seq_len:                       int
    tokens_by_dihedral:            Optional[List[torch.LongTensor]] = None
    cached_positions_by_dihedral:  Optional[List[torch.LongTensor]] = None
    seq_len_by_dihedral:           Optional[List[int]]              = None
    sep_index_by_dihedral:         Optional[List[int]]              = None


class PuzzleDataset(Dataset):

    _VALID_SPLITS = {"train", "test"}

    def __init__(
        self,
        json_path:            Path,
        splits:               Sequence[str] = ("train", "test"),
        include_outputs:      bool = True,
        max_seq_len:          int  = MAX_LEN,
        drop_long_sequences:  bool = False,
        task_whitelist:       Optional[Sequence[str]] = None,
        load_test_solutions:  bool = False,
    ) -> None:
        invalid = set(splits) - self._VALID_SPLITS
        if invalid:
            raise ValueError(f"Unsupported split(s) {invalid}. Expected: {self._VALID_SPLITS}.")

        self.source_path         = Path(json_path)
        self.max_seq_len         = max_seq_len
        self.drop_long_sequences = drop_long_sequences
        self.include_outputs     = include_outputs

        challenges   = load_tasks(self.source_path)
        solutions_map = self._load_solutions(load_test_solutions)

        task_ids = self._resolve_task_ids(task_whitelist, challenges, json_path)

        self.examples:               List[PuzzleSample]       = []
        self.task_id_to_example_id:  Dict[str, int]           = {}
        self.indices_by_split:       Dict[str, List[int]]     = {s: [] for s in splits}
        self.task_ids                                          = task_ids
        self.sequence_lengths:       List[int]                = []
        self.task_input_colors:      Dict[str, List[int]]     = {}

        for example_id, task_id in enumerate(task_ids):
            self.task_id_to_example_id[task_id] = example_id
            task = challenges[task_id]
            self.task_input_colors[task_id] = get_input_colors(task)

            for split in splits:
                for pair_index, pair in enumerate(task.get(split, [])):
                    self._process_pair(
                        task_id, task, split, pair, pair_index,
                        example_id, solutions_map, max_seq_len,
                        drop_long_sequences, include_outputs,
                    )

        self.num_examples = len(self.task_id_to_example_id)
        self._precompute_3d_positions()

    def _load_solutions(self, load_test_solutions: bool) -> Dict:
        if not load_test_solutions:
            return {}
        sol_path = self.source_path.with_name("solutions.json")
        if sol_path.exists():
            with sol_path.open("r") as f:
                return json.load(f)
        print(f"Warning: solutions.json not found at {sol_path}")
        return {}

    def _resolve_task_ids(self, whitelist, challenges, json_path) -> List[str]:
        if whitelist is None:
            return sorted(challenges.keys())
        missing = [tid for tid in whitelist if tid not in challenges]
        if missing:
            raise ValueError(f"Task ids {missing} were not found in {json_path}.")
        return list(whitelist)

    def _resolve_output_grid(self, pair, split, pair_index, task_id, solutions_map):
        output_grid = pair.get("output")
        if split == "test" and task_id in solutions_map:
            task_sols = solutions_map[task_id]
            if pair_index < len(task_sols):
                output_grid = task_sols[pair_index]
        return output_grid

    def _process_pair(
        self, task_id, task, split, pair, pair_index,
        example_id, solutions_map, max_seq_len,
        drop_long_sequences, include_outputs,
    ) -> None:
        input_grid  = pair["input"]
        output_grid = self._resolve_output_grid(pair, split, pair_index, task_id, solutions_map)

        has_output             = output_grid is not None
        include_output_tokens  = include_outputs and has_output
        append_end             = include_output_tokens

        tokens_by_dihedral:    List[torch.Tensor] = []
        seq_len_by_dihedral:   List[int]          = []
        sep_index_by_dihedral: List[int]          = []

        for transform_index in range(8):
            aug_input  = rotate_grid(input_grid, transform_index)
            aug_output = rotate_grid(output_grid, transform_index) if output_grid is not None else None

            tokens = tokenize_pair(
                aug_input, aug_output,
                include_output=include_output_tokens,
                append_end=append_end,
            )

            if len(tokens) > max_seq_len:
                if drop_long_sequences:
                    return
                raise ValueError(
                    f"Sequence length {len(tokens)} exceeds max_seq_len={max_seq_len} "
                    f"for task {task_id} ({split} pair {pair_index}) dihedral {transform_index}."
                )

            try:
                sep_index_by_dihedral.append(tokens.index(SEP_ID))
            except ValueError:
                raise ValueError(
                    f"Missing IO separator for task {task_id} ({split} pair {pair_index}) "
                    f"dihedral {transform_index}."
                )

            tokens_by_dihedral.append(torch.tensor(tokens, dtype=torch.long))
            seq_len_by_dihedral.append(len(tokens))

        example = PuzzleSample(
            tokens=tokens_by_dihedral[0],
            sep_index=sep_index_by_dihedral[0],
            example_id=example_id,
            task_id=task_id,
            split=split,
            pair_index=pair_index,
            has_output=has_output,
            seq_len=seq_len_by_dihedral[0],
            tokens_by_dihedral=tokens_by_dihedral,
            seq_len_by_dihedral=seq_len_by_dihedral,
            sep_index_by_dihedral=sep_index_by_dihedral,
        )
        self.indices_by_split.setdefault(split, []).append(len(self.examples))
        self.examples.append(example)
        self.sequence_lengths.append(max(seq_len_by_dihedral))

    def _precompute_3d_positions(self) -> None:
        print("Precomputing 3D positions...")
        for ex in self.examples:
            if ex.tokens_by_dihedral:
                cached: List[torch.Tensor] = []
                for tokens in ex.tokens_by_dihedral:
                    pos = get_3d_positions(tokens.unsqueeze(0), torch.ones(1, tokens.size(0), dtype=torch.bool))
                    cached.append(pos.squeeze(0))
                ex.cached_positions_by_dihedral = cached
                ex.cached_positions = cached[0]
            else:
                pos = get_3d_positions(ex.tokens.unsqueeze(0), torch.ones(1, ex.tokens.size(0), dtype=torch.bool))
                ex.cached_positions = pos.squeeze(0)

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> PuzzleSample:
        return self.examples[idx]

    def get_task_example_id(self, task_id: str) -> int:
        return self.task_id_to_example_id[task_id]

    def iter_examples(
        self,
        split:      Optional[str]  = None,
        has_output: Optional[bool] = None,
    ) -> Iterable[PuzzleSample]:
        for ex in self.examples:
            if split is not None and ex.split != split:
                continue
            if has_output is not None and ex.has_output != has_output:
                continue
            yield ex


# Bucket sampler

class BucketSampler(Sampler[List[int]]):

    def __init__(
        self,
        lengths:      Sequence[int],
        batch_size:   int,
        shuffle:      bool = True,
        bucket_size:  Optional[int] = None,
        drop_last:    bool = False,
    ) -> None:
        if batch_size <= 0:
            raise ValueError("batch_size must be positive.")
        self.lengths     = list(lengths)
        self.batch_size  = batch_size
        self.shuffle     = shuffle
        self.drop_last   = drop_last
        self.bucket_size = max(bucket_size or batch_size * 4, batch_size)

    def __len__(self) -> int:
        n = len(self.lengths)
        return n // self.batch_size if self.drop_last else (n + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        if not self.lengths:
            return iter(())

        if self.shuffle:
            indices = torch.randperm(len(self.lengths)).tolist()
        else:
            indices = sorted(range(len(self.lengths)), key=lambda i: self.lengths[i], reverse=True)

        batches: List[List[int]] = []
        if self.shuffle:
            for start in range(0, len(indices), self.bucket_size):
                bucket = sorted(indices[start: start + self.bucket_size], key=lambda i: self.lengths[i], reverse=True)
                for b_start in range(0, len(bucket), self.batch_size):
                    batch = bucket[b_start: b_start + self.batch_size]
                    if len(batch) == self.batch_size or not self.drop_last:
                        batches.append(batch)
            if len(batches) > 1:
                order = torch.randperm(len(batches)).tolist()
                batches = [batches[i] for i in order]
        else:
            for start in range(0, len(indices), self.batch_size):
                batch = indices[start: start + self.batch_size]
                if len(batch) == self.batch_size or not self.drop_last:
                    batches.append(batch)

        return iter(batches)


# Collation

def batch_collate(
    batch: List[PuzzleSample],
    pad_token_id: int = END_ID,
    augment_selector: Optional[
        Callable[[PuzzleSample], Tuple[Optional[torch.Tensor], Optional[int]]]
    ] = None,
) -> Dict[str, torch.Tensor]:
    if not batch:
        raise ValueError("Empty batch encountered during collation.")
    del pad_token_id

    def _resolve_example(ex: PuzzleSample):
        tokens         = ex.tokens
        sep_index      = ex.sep_index
        cached_pos     = getattr(ex, "cached_positions", None)
        mapping        = None
        dihedral_id    = 0

        if augment_selector is not None:
            mapping, transform_index = augment_selector(ex)
            if transform_index is not None:
                if not (0 <= transform_index < 8):
                    raise ValueError(f"Invalid dihedral index {transform_index} for {ex.task_id}.")
                dihedral_id = int(transform_index)
                if ex.tokens_by_dihedral:
                    tokens    = ex.tokens_by_dihedral[transform_index]
                    cached_pos = (ex.cached_positions_by_dihedral or [None] * 8)[transform_index]
                    sep_index  = (ex.sep_index_by_dihedral or [sep_index] * 8)[transform_index]

        return tokens, cached_pos, int(tokens.size(0)), sep_index, mapping, dihedral_id

    resolved = [_resolve_example(ex) for ex in batch]
    batch_size   = len(resolved)
    seq_lengths  = torch.tensor([r[2] for r in resolved], dtype=torch.int32)
    max_len      = int(seq_lengths.max().item())
    total_tokens = int(seq_lengths.sum().item())

    input_ids    = torch.zeros(total_tokens, dtype=torch.long)
    example_ids  = torch.zeros(batch_size,   dtype=torch.long)
    dihedral_ids = torch.zeros(batch_size,   dtype=torch.long)
    sep_indices  = torch.zeros(batch_size,   dtype=torch.long)
    positions_3d = torch.zeros((total_tokens, 3), dtype=torch.long)

    cursor = 0
    for idx, (ex, (tokens, cached_pos, seq_len, sep_index, mapping, dihedral_id)) in enumerate(zip(batch, resolved)):
        if mapping is not None:
            tokens = mapping[tokens]
        end = cursor + seq_len
        input_ids[cursor:end] = tokens
        example_ids[idx]      = ex.example_id
        dihedral_ids[idx]     = dihedral_id
        sep_indices[idx]      = sep_index
        if cached_pos is None:
            fake = tokens.unsqueeze(0)
            cached_pos = get_3d_positions(fake, torch.ones_like(fake, dtype=torch.bool)).squeeze(0)
        positions_3d[cursor:end] = cached_pos
        cursor = end

    cu_seqlens = torch.zeros(batch_size + 1, dtype=torch.int32)
    cu_seqlens[1:] = torch.cumsum(seq_lengths, dim=0)

    return {
        "input_ids":   input_ids,
        "cu_seqlens":  cu_seqlens,
        "max_seqlen":  max_len,
        "has_padding": False,
        "example_ids": example_ids,
        "dihedral_ids": dihedral_ids,
        "sep_indices": sep_indices,
        "positions_3d": positions_3d,
        "task_ids":    [ex.task_id for ex in batch],
        "splits":      [ex.split for ex in batch],
        "has_output":  [ex.has_output for ex in batch],
    }


def make_dataloader(
    dataset:                  PuzzleDataset,
    batch_size:               int,
    shuffle:                  bool = True,
    bucket_size_multiplier:   int  = 4,
    augment_selector:         Optional[Callable] = None,
    use_length_bucketing:     bool = True,
) -> DataLoader:
    collate_fn = (
        functools.partial(batch_collate, augment_selector=augment_selector)
        if augment_selector is not None
        else batch_collate
    )

    if use_length_bucketing:
        lengths = getattr(dataset, "sequence_lengths", None) or [
            len(dataset[i].tokens) for i in range(len(dataset))
        ]
        bucket_size = max(batch_size * max(1, bucket_size_multiplier), batch_size)
        return DataLoader(
            dataset,
            batch_sampler=BucketSampler(lengths, batch_size, shuffle=shuffle, bucket_size=bucket_size),
            num_workers=0,
            collate_fn=collate_fn,
        )

    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        drop_last=False, num_workers=0, collate_fn=collate_fn,
    )


# Utility: seeds & device

def fix_seed(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device(device_str: str) -> torch.device:
    device_str = str(device_str or "cuda").lower()
    if not device_str.startswith("cuda"):
        raise ValueError("Only CUDA is supported. Set device='cuda'.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required but not available.")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    return torch.device(device_str)


# RNG snapshot/restore (for deterministic checkpoint resumption)

def save_rng(device: torch.device) -> Dict[str, Any]:
    state: Dict[str, Any] = {
        "python": random.getstate(),
        "numpy":  np.random.get_state(),
        "torch":  torch.get_rng_state(),
    }
    if torch.cuda.is_available() and device.type == "cuda":
        try:
            state["cuda"] = torch.cuda.get_rng_state_all()
        except Exception:
            pass
    return state


def load_rng(state: Optional[Dict[str, Any]], device: torch.device) -> None:
    if not state:
        return
    for key, restore_fn in [
        ("python", lambda s: random.setstate(s)),
        ("numpy",  lambda s: np.random.set_state(s)),
        ("torch",  lambda s: torch.set_rng_state(s)),
    ]:
        try:
            restore_fn(state[key])
        except Exception:
            pass
    if "cuda" in state and torch.cuda.is_available() and device.type == "cuda":
        try:
            torch.cuda.set_rng_state_all(state["cuda"])
        except Exception:
            pass


# Augmentation internals

def _extract_input_tokens(tokens: Sequence[int]) -> List[int]:
    return [int(tok) for tok in tokens if int(tok) != SEP_ID] if SEP_ID not in tokens else \
           list(tokens)[:tokens.index(SEP_ID)] if isinstance(tokens, list) else \
           [int(tok) for tok in tokens if int(tok) != SEP_ID]


def _extract_input_tokens(tokens: Sequence[int]) -> List[int]:
    result: List[int] = []
    for tok in tokens:
        val = int(tok)
        if val == SEP_ID:
            break
        result.append(val)
    return result


def _hash_tokens(tokens: Sequence[int]) -> int:
    digest = hashlib.sha256(bytes(tokens)).digest()
    return int.from_bytes(digest[:8], "little")


def _derive_order_seed(seed: int, key: Tuple[str, str, int]) -> int:
    payload = f"{int(seed)}|{key[0]}|{key[1]}|{int(key[2])}"
    digest  = hashlib.sha256(payload.encode()).digest()
    return int.from_bytes(digest[:8], "little")


def _cycle_seed(order_seed: int, mode: int, cycle: int) -> int:
    payload = f"{int(order_seed)}|{int(mode)}|{int(cycle)}"
    digest  = hashlib.sha256(payload.encode()).digest()
    return int.from_bytes(digest[:8], "little")


def _shuffled_indices(indices: Sequence[int], seed: int) -> List[int]:
    order = list(indices)
    random.Random(int(seed)).shuffle(order)
    return order


def _index_for_epoch(order_seed: int, indices: Sequence[int], epoch: int, mode: int) -> int:
    if len(indices) == 1:
        return int(indices[0])
    epoch_i   = max(0, int(epoch))
    cycle_len = len(indices)
    cycle     = epoch_i // cycle_len
    offset    = epoch_i % cycle_len
    order     = _shuffled_indices(indices, _cycle_seed(order_seed, mode, cycle))
    if cycle > 0:
        prev = _shuffled_indices(indices, _cycle_seed(order_seed, mode, cycle - 1))
        if order == prev:
            order = order[1:] + order[:1]
    return int(order[offset])


def _colors_from_tokens(tokens: Sequence[int]) -> List[int]:
    return sorted({int(tok) for tok in tokens if 1 <= int(tok) <= 9})


def _max_color_permutations(n_colors: int, k: int) -> int:
    if k <= 0 or n_colors <= 0 or k > n_colors:
        return 1
    return math.factorial(n_colors) // math.factorial(n_colors - k)


def _unrank_permutation(domain: Sequence[int], index: int, factorials: Sequence[int]) -> Tuple[int, ...]:
    items  = [int(c) for c in domain]
    result: List[int] = []
    idx    = int(index)
    for size in range(len(items), 0, -1):
        pos = idx // factorials[size - 1]
        idx %= factorials[size - 1]
        result.append(int(items.pop(pos)))
    return tuple(result)


def _generate_task_permutations(domain: Sequence[int], max_permutations: int, rng: random.Random) -> List[Tuple[int, ...]]:
    domain_list = [int(c) for c in domain]
    n = len(domain_list)
    if n == 0:
        return [tuple()]
    total  = math.factorial(n)
    target = min(max(1, int(max_permutations)), total)
    if target == 1:
        return [tuple(domain_list)]
    factorials = [1]
    for i in range(1, n + 1):
        factorials.append(factorials[-1] * i)
    if target >= total:
        indices = list(range(total))
        rng.shuffle(indices)
    else:
        indices = rng.sample(range(1, total), target - 1) + [0]
        rng.shuffle(indices)
    return [_unrank_permutation(domain_list, idx, factorials) for idx in indices]


def _mapping_from_permutation(domain: Sequence[int], perm: Sequence[int]) -> List[int]:
    mapping = list(range(N_VOCAB))
    for src, dst in zip(domain, perm):
        mapping[int(src)] = int(dst)
    return mapping


# Augmentor

@dataclass
class Augments:
    color_maps:              torch.Tensor
    dihedral_indices:        List[int]
    color_map_indices:       List[int]
    identity_tuple_index:    int
    color_identity_indices:  List[int]
    dihedral_identity_indices: List[int]
    order_seed:              int = 0


class Augmentor:

    def __init__(
        self,
        augments_by_key:            Dict[Tuple[str, str, int], Augments],
        *,
        color_apply_to_test_split:   bool,
        dihedral_apply_to_test_split: bool,
        max_augments:                int = 0,
    ) -> None:
        self.augments_by_key             = augments_by_key
        self.color_apply_to_test_split   = bool(color_apply_to_test_split)
        self.dihedral_apply_to_test_split = bool(dihedral_apply_to_test_split)
        self.max_augments                = int(max_augments)
        self._enabled                    = True
        self._epoch                      = 0

    def set_enabled(self, enabled: bool) -> None:
        self._enabled = bool(enabled)

    def set_epoch(self, epoch: int) -> None:
        self._epoch = max(0, int(epoch))

    def _augment_flags_for_split(self, split: str) -> Tuple[bool, bool]:
        if split == "test":
            return self.color_apply_to_test_split, self.dihedral_apply_to_test_split
        return True, True

    def _select_index_for_epoch(
        self, augments: Augments, *, color_enabled: bool, dihedral_enabled: bool
    ) -> int:
        if not color_enabled and not dihedral_enabled:
            return augments.identity_tuple_index
        if not color_enabled:
            candidates = augments.color_identity_indices
        elif not dihedral_enabled:
            candidates = augments.dihedral_identity_indices
        else:
            candidates = list(range(len(augments.dihedral_indices)))
        if not candidates:
            return augments.identity_tuple_index
        if len(candidates) == 1:
            return candidates[0]
        mode = (1 if color_enabled else 0) | (2 if dihedral_enabled else 0)
        return _index_for_epoch(augments.order_seed, candidates, self._epoch, mode)

    def select_for_example(self, example: PuzzleSample) -> Tuple[Optional[torch.Tensor], Optional[int]]:
        if not self._enabled:
            return None, None
        key     = (example.task_id, example.split, example.pair_index)
        augments = self.augments_by_key.get(key)
        if augments is None:
            return None, None
        color_en, dihedral_en = self._augment_flags_for_split(example.split)
        idx        = self._select_index_for_epoch(augments, color_enabled=color_en, dihedral_enabled=dihedral_en)
        mapping    = augments.color_maps[augments.color_map_indices[idx]]
        dihedral   = augments.dihedral_indices[idx]
        return mapping, dihedral


def setup_augmentor(
    dataset:                        Iterable[PuzzleSample],
    task_input_colors:              Dict[str, Sequence[int]],
    *,
    max_augments:                   int,
    enable_color:                   bool,
    enable_dihedral:                bool,
    seed:                           int,
    color_apply_to_test_split:      bool,
    dihedral_apply_to_test_split:   bool,
) -> Augmentor:
    rng          = random.Random(int(seed))
    max_augments = max(1, int(max_augments))
    max_dihedral = 8 if enable_dihedral else 1

    examples_by_task:       Dict[str, List[Tuple[str, str, int]]] = {}
    input_tokens_by_key:    Dict[Tuple[str, str, int], List[List[int]]] = {}
    input_colors_by_key:    Dict[Tuple[str, str, int], List[int]] = {}

    for ex in dataset:
        key             = (ex.task_id, ex.split, ex.pair_index)
        tokens_list     = ex.tokens_by_dihedral or [ex.tokens]
        num_d           = 8 if enable_dihedral else 1
        inp_tokens_per_d = []
        for d in range(num_d):
            toks = tokens_list[d]
            inp_tokens_per_d.append(_extract_input_tokens(toks.tolist() if isinstance(toks, torch.Tensor) else list(toks)))
        input_tokens_by_key[key] = inp_tokens_per_d
        input_colors_by_key[key] = _colors_from_tokens(inp_tokens_per_d[0])
        examples_by_task.setdefault(ex.task_id, []).append(key)

    augments_by_key: Dict[Tuple[str, str, int], Augments] = {}
    stats: List[int] = []

    for task_id, keys in examples_by_task.items():
        seen_hashes: Set[int] = set()
        for key in keys:
            seen_hashes.add(_hash_tokens(input_tokens_by_key[key][0]))

        allowed_colors  = [int(c) for c in task_input_colors.get(task_id, []) if 1 <= int(c) <= 9]
        task_input_set  = set().union(*[set(input_colors_by_key[k]) for k in keys])
        colors_a        = sorted(task_input_set)
        allowed_set     = set(allowed_colors) or set(colors_a)
        domain_colors   = colors_a + sorted(allowed_set - set(colors_a))
        identity_perm   = tuple(domain_colors)
        identity_map    = list(range(N_VOCAB))

        sequence_states: Dict[Tuple[str, str, int], Dict[str, object]] = {}
        remaining = 0

        for key in keys:
            input_colors   = input_colors_by_key[key]
            max_color_x    = _max_color_permutations(len(domain_colors), len(input_colors))
            max_possible   = max_color_x * max_dihedral if enable_color else max_dihedral
            augment_limit  = max(1, min(max_augments, max_possible))
            order_seed     = _derive_order_seed(seed, key)
            dihedral_order = _shuffled_indices(list(range(max_dihedral)), order_seed) if max_dihedral > 1 else [0]
            identity_sig   = tuple(input_colors)

            state: Dict[str, object] = {
                "input_colors":            input_colors,
                "input_tokens_by_dihedral": input_tokens_by_key[key],
                "color_maps":              [identity_map],
                "dihedral_indices":        [0],
                "color_map_indices":       [0],
                "augment_limit":           augment_limit,
                "augment_count":           1,
                "seen_signatures":         {identity_sig},
                "seen_pairs":              {(identity_sig, 0)},
                "mapping_index_by_key":    {tuple(identity_map): 0},
                "mapping_signatures":      [identity_sig],
                "dihedral_order":          dihedral_order,
                "dihedral_cursor":         0,
                "order_seed":              order_seed,
            }
            if state["augment_count"] < state["augment_limit"]:
                remaining += 1
            sequence_states[key] = state

        if enable_color and remaining > 0:
            task_permutations = _generate_task_permutations(domain_colors, 5000, rng)
            for perm in task_permutations:
                if remaining == 0:
                    break
                if perm == identity_perm:
                    continue
                mapping     = _mapping_from_permutation(domain_colors, perm)
                mapping_key = tuple(mapping)

                for key in keys:
                    state = sequence_states[key]
                    if state["augment_count"] >= state["augment_limit"]:
                        continue
                    input_colors    = state["input_colors"]
                    signature       = tuple(mapping[c] for c in input_colors)
                    if signature in state["seen_signatures"]:
                        continue
                    inp_toks        = state["input_tokens_by_dihedral"]
                    d_order         = state["dihedral_order"]
                    d_cursor        = int(state["dihedral_cursor"])
                    order_len       = len(d_order)
                    selected_idx    = selected_hash = None

                    for offset in range(order_len):
                        d_idx = int(d_order[(d_cursor + offset) % order_len])
                        if (signature, d_idx) in state["seen_pairs"]:
                            continue
                        mapped = [mapping[tok] if 0 <= tok < len(mapping) else tok for tok in inp_toks[d_idx]]
                        hashed = _hash_tokens(mapped)
                        state["seen_pairs"].add((signature, d_idx))
                        if hashed in seen_hashes:
                            continue
                        selected_idx  = d_idx
                        selected_hash = hashed
                        state["dihedral_cursor"] = (d_cursor + offset + 1) % order_len
                        break

                    if selected_idx is None:
                        state["seen_signatures"].add(signature)
                        continue

                    map_idx = state["mapping_index_by_key"].get(mapping_key)
                    if map_idx is None:
                        map_idx = len(state["color_maps"])
                        state["color_maps"].append(mapping)
                        state["mapping_index_by_key"][mapping_key] = map_idx
                        state["mapping_signatures"].append(signature)
                    state["dihedral_indices"].append(int(selected_idx))
                    state["color_map_indices"].append(int(map_idx))
                    state["augment_count"] += 1
                    state["seen_signatures"].add(signature)
                    seen_hashes.add(selected_hash)
                    if state["augment_count"] >= state["augment_limit"]:
                        remaining -= 1
                        if remaining == 0:
                            break

        if enable_dihedral and remaining > 0:
            for pass_idx in range(max_dihedral):
                if remaining == 0:
                    break
                for key in keys:
                    state = sequence_states[key]
                    if state["augment_count"] >= state["augment_limit"]:
                        continue
                    inp_toks   = state["input_tokens_by_dihedral"]
                    d_order    = _shuffled_indices(list(range(max_dihedral)), _cycle_seed(state["order_seed"], 3, pass_idx))
                    for map_idx, signature in enumerate(state["mapping_signatures"]):
                        if state["augment_count"] >= state["augment_limit"]:
                            break
                        mapping = state["color_maps"][map_idx]
                        for d in d_order:
                            if state["augment_count"] >= state["augment_limit"]:
                                break
                            if (signature, d) in state["seen_pairs"]:
                                continue
                            mapped = [mapping[tok] if 0 <= tok < len(mapping) else tok for tok in inp_toks[d]]
                            hashed = _hash_tokens(mapped)
                            state["seen_pairs"].add((signature, d))
                            if hashed in seen_hashes:
                                continue
                            state["dihedral_indices"].append(int(d))
                            state["color_map_indices"].append(int(map_idx))
                            state["augment_count"] += 1
                            seen_hashes.add(hashed)
                            if state["augment_count"] >= state["augment_limit"]:
                                remaining -= 1
                            break
                    if remaining == 0:
                        break

        for key in keys:
            state = sequence_states[key]
            augments_by_key[key] = Augments(
                color_maps=torch.tensor(state["color_maps"], dtype=torch.long),
                dihedral_indices=state["dihedral_indices"],
                color_map_indices=state["color_map_indices"],
                identity_tuple_index=0,
                color_identity_indices=[i for i, c in enumerate(state["color_map_indices"]) if c == 0],
                dihedral_identity_indices=[i for i, d in enumerate(state["dihedral_indices"]) if d == 0],
                order_seed=_derive_order_seed(seed, key),
            )
            stats.append(len(state["dihedral_indices"]))

    if stats:
        avg = sum(stats) / len(stats)
        print(f"Augmentation stats: {len(stats)} sequences, avg augments={avg:.1f}, min={min(stats)}, max={max(stats)}")

    return Augmentor(
        augments_by_key,
        color_apply_to_test_split=color_apply_to_test_split,
        dihedral_apply_to_test_split=dihedral_apply_to_test_split,
        max_augments=max_augments,
    )

## Transformer Architecture


In [4]:
# ── Model Configuration ────────────────────────────────────────────────────────

@dataclass
class ARCTransformerConfig:
    """Hyperparameters for the ARC Transformer model."""
    vocab_size:        int   = N_VOCAB
    max_seq_len:       int   = MAX_LEN   # 512 — governs positional embedding range
    d_model:           int   = 640       # default matches ~49.7M architecture
    n_heads:           int   = 10
    d_ff:              int   = 2560
    n_layers:          int   = 10
    dropout:           float = 0.1
    attention_dropout: Optional[float] = None
    num_examples:      int   = 1280
    num_dihedrals:     int   = 8

    def __post_init__(self) -> None:
        if self.attention_dropout is None:
            self.attention_dropout = self.dropout
        if self.d_model % self.n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")
        if self.n_layers < 1:
            raise ValueError("n_layers must be >= 1.")
        if self.num_examples < 1:
            raise ValueError("num_examples must be >= 1.")
        if self.num_dihedrals != 8:
            raise ValueError("num_dihedrals must be exactly 8.")


# ── Normalisation ──────────────────────────────────────────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return x * torch.rsqrt(rms + self.eps) * self.weight


# ── 3-D Rotary Positional Embedding ───────────────────────────────────────────

class RoPE3D(nn.Module):
    """3D Rotary Positional Embedding applied to Q/K projections.

    Splits head_dim into three even slices (x, y, z) and precomputes cos/sin
    lookup tables for fixed coordinate ranges (x: 0–32, y: 0–32, z: 0–8).
    """

    def __init__(self, head_dim: int, base: float = 10000.0) -> None:
        super().__init__()
        if head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")
        self.head_dim = head_dim
        self.base     = base

        n_pairs = head_dim // 2
        px = n_pairs // 3
        py = n_pairs // 3
        pz = n_pairs - px - py
        self.d_x = px * 2
        self.d_y = py * 2
        self.d_z = pz * 2

        self.max_x = 32
        self.max_y = 32
        self.max_z = 8

        self._register_cache("x", self.d_x, self.max_x)
        self._register_cache("y", self.d_y, self.max_y)
        self._register_cache("z", self.d_z, self.max_z)

    def _build_inv_freq(self, dim: int) -> torch.Tensor:
        if dim <= 0:
            return torch.empty(0)
        return 1.0 / (self.base ** (torch.arange(0, dim, 2).float() / dim))

    def _register_cache(self, name: str, dim: int, max_pos: int) -> None:
        if dim <= 0:
            self.register_buffer(f"cos_{name}_cache", torch.empty(0), persistent=False)
            self.register_buffer(f"sin_{name}_cache", torch.empty(0), persistent=False)
            return
        inv_freq = self._build_inv_freq(dim)
        pos      = torch.arange(max_pos).float()
        t        = pos.unsqueeze(-1) * inv_freq
        self.register_buffer(f"cos_{name}_cache", torch.cos(t).repeat_interleave(2, dim=-1), persistent=False)
        self.register_buffer(f"sin_{name}_cache", torch.sin(t).repeat_interleave(2, dim=-1), persistent=False)

    @staticmethod
    def _rotate_half(x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def apply_rotary(
        self, q: torch.Tensor, k: torch.Tensor, pos_xyz: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Apply 3D RoPE.

        Args:
            q, k:    [B, H, S, D]
            pos_xyz: [B, S, 3] integer coordinates (x, y, z)
        """
        pos_x = pos_xyz[..., 0].clamp(0, self.max_x - 1)
        pos_y = pos_xyz[..., 1].clamp(0, self.max_y - 1)
        pos_z = pos_xyz[..., 2].clamp(0, self.max_z - 1)

        parts_cos, parts_sin = [], []
        if self.d_x > 0:
            parts_cos.append(self.cos_x_cache[pos_x])
            parts_sin.append(self.sin_x_cache[pos_x])
        if self.d_y > 0:
            parts_cos.append(self.cos_y_cache[pos_y])
            parts_sin.append(self.sin_y_cache[pos_y])
        if self.d_z > 0:
            parts_cos.append(self.cos_z_cache[pos_z])
            parts_sin.append(self.sin_z_cache[pos_z])

        cos = torch.cat(parts_cos, dim=-1).unsqueeze(1)  # [B, 1, S, D]
        sin = torch.cat(parts_sin, dim=-1).unsqueeze(1)

        q_out = (q * cos) + (self._rotate_half(q) * sin)
        k_out = (k * cos) + (self._rotate_half(k) * sin)
        return q_out, k_out


# ── Self-Attention ─────────────────────────────────────────────────────────────

class SelfAttention(nn.Module):
    def __init__(self, config: ARCTransformerConfig) -> None:
        super().__init__()
        self.n_heads   = config.n_heads
        self.head_dim  = config.d_model // config.n_heads
        self.scale     = self.head_dim ** -0.5
        self.dropout_p = float(config.attention_dropout)

        self._has_flash_attn_varlen = flash_attn_varlen_qkvpacked_func is not None
        self._has_flex_attention    = flex_attention is not None

        self.qkv_proj = nn.Linear(config.d_model, 3 * config.d_model, bias=False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.rope     = RoPE3D(self.head_dim)

    # ── Attention bias construction ────────────────────────────────────────────

    def _build_sdpa_attn_bias(
        self,
        queries:        torch.Tensor,
        keys:           torch.Tensor,
        attention_mask: Optional[torch.Tensor],
        causal_mask:    Optional[torch.Tensor],
        sdpa_mask:      Optional[torch.Tensor],
    ) -> Optional[torch.Tensor]:
        if sdpa_mask is not None:
            return sdpa_mask

        batch_size = queries.size(0)
        query_len  = queries.size(2)
        key_len    = keys.size(2)
        attn_bias  = None

        if causal_mask is not None:
            attn_bias = torch.zeros((1, 1, query_len, key_len), device=queries.device, dtype=queries.dtype)
            if causal_mask.size(-2) == query_len and causal_mask.size(-1) == key_len:
                attn_bias = attn_bias.masked_fill(causal_mask, float("-inf"))
            else:
                q_pos = torch.arange(query_len, device=queries.device).view(-1, 1)
                k_pos = torch.arange(key_len,   device=queries.device).view(1, -1)
                attn_bias = attn_bias.masked_fill((q_pos < k_pos)[None, None], float("-inf"))

        if attention_mask is not None:
            if attention_mask.size(1) < key_len:
                raise ValueError("attention_mask length must cover all key/value positions.")
            if attn_bias is None:
                attn_bias = torch.zeros((batch_size, 1, query_len, key_len), device=queries.device, dtype=queries.dtype)
            attn_bias = attn_bias.masked_fill(~attention_mask[:, None, None, :key_len], float("-inf"))

        return attn_bias

    # ── Attention dispatch helpers ─────────────────────────────────────────────

    def _apply_attention(
        self,
        queries:        torch.Tensor,
        keys:           torch.Tensor,
        values:         torch.Tensor,
        attention_mask: Optional[torch.Tensor],
        causal_mask:    Optional[torch.Tensor],
        sdpa_mask:      Optional[torch.Tensor],
        dropout_p:      float,
        is_causal:      bool,
    ) -> torch.Tensor:
        use_builtin_causal = bool(is_causal or causal_mask is not None)
        if sdpa_mask is None and attention_mask is None:
            return F.scaled_dot_product_attention(
                queries, keys, values,
                attn_mask=None, dropout_p=dropout_p, is_causal=use_builtin_causal,
            )
        attn_bias = self._build_sdpa_attn_bias(queries, keys, attention_mask, causal_mask, sdpa_mask)
        return F.scaled_dot_product_attention(
            queries, keys, values, attn_mask=attn_bias, dropout_p=dropout_p, is_causal=False,
        )

    def _apply_flex_decode_attention(
        self,
        queries:          torch.Tensor,
        keys:             torch.Tensor,
        values:           torch.Tensor,
        attention_mask:   torch.Tensor,
        decode_block_mask: Optional[object] = None,
    ) -> torch.Tensor:
        if not self._has_flex_attention:
            raise RuntimeError("flex_attention is not available.")
        if attention_mask.dim() != 2:
            raise ValueError("Decode attention_mask must have shape [batch, key_len].")
        if attention_mask.size(0) != queries.size(0):
            raise ValueError("Decode attention_mask batch size must match query batch size.")
        if attention_mask.size(1) < keys.size(2):
            raise ValueError("Decode attention_mask must cover all key/value positions.")

        key_len = keys.size(2)
        mask    = attention_mask[:, :key_len]

        if decode_block_mask is not None:
            return flex_attention(queries, keys, values, block_mask=decode_block_mask, scale=self.scale)

        # Fallback: score_mod when create_block_mask is unavailable
        def decode_score_mod(score, b, h, q_idx, kv_idx):
            del h, q_idx
            return torch.where(mask[b, kv_idx], score, score.new_full((), float("-inf")))

        return flex_attention(queries, keys, values, score_mod=decode_score_mod, scale=self.scale)

    def _apply_varlen_flash_attention(
        self,
        qkv:        torch.Tensor,
        cu_seqlens: torch.Tensor,
        max_seqlen: int,
        dropout_p:  float,
        is_causal:  bool,
    ) -> torch.Tensor:
        """Causal attention over packed (variable-length) sequences.

        Uses flash-attn when available; falls back to per-sequence SDPA otherwise.
        """
        if self._has_flash_attn_varlen and qkv.device.type == "cuda":
            if cu_seqlens.device != qkv.device or cu_seqlens.dtype != torch.int32:
                cu_seqlens = cu_seqlens.to(device=qkv.device, dtype=torch.int32)
            return flash_attn_varlen_qkvpacked_func(
                qkv, cu_seqlens, int(max_seqlen),
                dropout_p=dropout_p, softmax_scale=None, causal=is_causal,
            )

        # SDPA fallback: unpack each sequence, compute attention, re-pack
        cu        = cu_seqlens.to(dtype=torch.long, device=qkv.device)
        starts    = cu[:-1]
        ends      = cu[1:]
        batch_sz  = starts.size(0)
        dtype     = qkv.dtype
        output    = torch.empty(qkv.size(0), self.n_heads, self.head_dim, dtype=dtype, device=qkv.device)

        for i in range(batch_sz):
            s, e = int(starts[i].item()), int(ends[i].item())
            q = qkv[s:e, 0].transpose(0, 1).unsqueeze(0)
            k = qkv[s:e, 1].transpose(0, 1).unsqueeze(0)
            v = qkv[s:e, 2].transpose(0, 1).unsqueeze(0)
            out = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=dropout_p if self.training else 0.0,
                is_causal=is_causal,
            )
            output[s:e] = out.squeeze(0).transpose(0, 1)

        return output

    # ── Forward (packed varlen or batched) ────────────────────────────────────

    def forward(
        self,
        hidden_states:  torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        causal_mask:    Optional[torch.Tensor] = None,
        pos_xyz:        Optional[torch.Tensor] = None,
        sdpa_mask:      Optional[torch.Tensor] = None,
        is_causal:      bool = False,
        cu_seqlens:     Optional[torch.Tensor] = None,
        max_seqlen:     Optional[int] = None,
    ) -> torch.Tensor:
        if hidden_states.dim() == 2:
            return self._forward_varlen(hidden_states, pos_xyz, cu_seqlens, max_seqlen)
        if hidden_states.dim() != 3:
            raise ValueError("hidden_states must be rank-2 (packed) or rank-3 (batched).")
        return self._forward_batched(
            hidden_states, attention_mask, causal_mask, pos_xyz, sdpa_mask, is_causal
        )

    def _forward_varlen(
        self,
        hidden_states: torch.Tensor,
        pos_xyz:       Optional[torch.Tensor],
        cu_seqlens:    Optional[torch.Tensor],
        max_seqlen:    Optional[int],
    ) -> torch.Tensor:
        if cu_seqlens is None:
            raise ValueError("cu_seqlens is required for packed varlen attention inputs.")
        if max_seqlen is None:
            raise ValueError("max_seqlen is required for packed varlen attention inputs.")

        total_tokens, dim = hidden_states.shape
        qkv = self.qkv_proj(hidden_states).view(total_tokens, 3, self.n_heads, self.head_dim)
        queries, keys, values = qkv.unbind(1)
        dtype = queries.dtype

        if pos_xyz is not None:
            if pos_xyz.dim() != 2 or pos_xyz.size(0) != total_tokens:
                raise ValueError("Packed pos_xyz must have shape [total_tokens, 3].")
            q_f32 = queries.transpose(0, 1).unsqueeze(0).float()
            k_f32 = keys.transpose(0, 1).unsqueeze(0).float()
            q_f32, k_f32 = self.rope.apply_rotary(q_f32, k_f32, pos_xyz.unsqueeze(0))
            queries = q_f32.squeeze(0).transpose(0, 1).to(dtype=dtype)
            keys    = k_f32.squeeze(0).transpose(0, 1).to(dtype=dtype)

        qkv_packed = torch.stack((queries, keys, values), dim=1).contiguous()
        attn_out   = self._apply_varlen_flash_attention(
            qkv=qkv_packed, cu_seqlens=cu_seqlens, max_seqlen=max_seqlen,
            dropout_p=self.dropout_p if self.training else 0.0, is_causal=True,
        )
        return self.out_proj(attn_out.contiguous().view(total_tokens, dim))

    def _forward_batched(
        self,
        hidden_states:  torch.Tensor,
        attention_mask: Optional[torch.Tensor],
        causal_mask:    Optional[torch.Tensor],
        pos_xyz:        Optional[torch.Tensor],
        sdpa_mask:      Optional[torch.Tensor],
        is_causal:      bool,
    ) -> torch.Tensor:
        batch_size, seq_len, dim = hidden_states.shape
        qkv     = self.qkv_proj(hidden_states).view(batch_size, seq_len, 3, self.n_heads, self.head_dim)
        qkv     = qkv.permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv.unbind(0)
        dtype   = queries.dtype

        if pos_xyz is not None:
            q_f32, k_f32 = self.rope.apply_rotary(queries.float(), keys.float(), pos_xyz)
            queries = q_f32.to(dtype=dtype)
            keys    = k_f32.to(dtype=dtype)

        attn_out = self._apply_attention(
            queries, keys, values,
            attention_mask=attention_mask, causal_mask=causal_mask,
            sdpa_mask=sdpa_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, dim)
        return self.out_proj(attn_out)

    # ── KV-cache forward (generation) ────────────────────────────────────────

    def forward_with_cache(
        self,
        hidden_states:     torch.Tensor,
        pos_xyz:           torch.Tensor,
        attention_mask:    Optional[torch.Tensor] = None,
        causal_mask:       Optional[torch.Tensor] = None,
        past_key_value:    Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        cache_position:    Optional[torch.Tensor] = None,
        decode_block_mask: Optional[object] = None,
    ) -> torch.Tensor:
        batch_size, seq_len, dim = hidden_states.shape
        qkv     = self.qkv_proj(hidden_states).view(batch_size, seq_len, 3, self.n_heads, self.head_dim)
        qkv     = qkv.permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv.unbind(0)
        dtype   = queries.dtype

        if pos_xyz is not None:
            q_f32, k_f32 = self.rope.apply_rotary(queries.float(), keys.float(), pos_xyz)
            queries = q_f32.to(dtype=dtype)
            keys    = k_f32.to(dtype=dtype)

        if past_key_value is not None:
            # Decode step: write new K/V into the pre-allocated cache buffers
            past_keys, past_values = past_key_value
            if cache_position is not None:
                past_keys.index_copy_(2, cache_position, keys)
                past_values.index_copy_(2, cache_position, values)

            use_flex = (
                self._has_flex_attention
                and queries.device.type == "cuda"
                and attention_mask is not None
            )
            if use_flex:
                attn_out = self._apply_flex_decode_attention(
                    queries, past_keys, past_values,
                    attention_mask=attention_mask, decode_block_mask=decode_block_mask,
                )
            else:
                attn_out = self._apply_attention(
                    queries, past_keys, past_values,
                    attention_mask=attention_mask, causal_mask=None,
                    sdpa_mask=None, dropout_p=0.0, is_causal=False,
                )
            attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, dim)
            return self.out_proj(attn_out)

        # Prompt pass: return output AND the new KV tensors to fill the cache
        attn_out = self._apply_attention(
            queries, keys, values,
            attention_mask=attention_mask, causal_mask=causal_mask,
            sdpa_mask=None, dropout_p=0.0, is_causal=causal_mask is not None,
        )
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, dim)
        return self.out_proj(attn_out), (keys, values)


# ── Feed-Forward Network ───────────────────────────────────────────────────────

class FFN(nn.Module):
    """SwiGLU feed-forward network."""

    def __init__(self, config: ARCTransformerConfig) -> None:
        super().__init__()
        self.fc_in   = nn.Linear(config.d_model, config.d_ff * 2)
        self.fc_out  = nn.Linear(config.d_ff, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate, hidden = self.fc_in(x).chunk(2, dim=-1)
        x = gate * F.silu(hidden)
        return self.dropout(self.fc_out(self.dropout(x)))


# ── Transformer Block ──────────────────────────────────────────────────────────

class Block(nn.Module):
    """Pre-norm transformer block (RMSNorm + SelfAttention + FFN)."""

    def __init__(self, config: ARCTransformerConfig) -> None:
        super().__init__()
        self.ln_1      = RMSNorm(config.d_model)
        self.attention = SelfAttention(config)
        self.ln_2      = RMSNorm(config.d_model)
        self.ff        = FFN(config)

    def forward(
        self,
        hidden_states:  torch.Tensor,
        attention_mask: Optional[torch.Tensor],
        causal_mask:    Optional[torch.Tensor],
        pos_xyz:        Optional[torch.Tensor],
        sdpa_mask:      Optional[torch.Tensor] = None,
        is_causal:      bool = False,
        cu_seqlens:     Optional[torch.Tensor] = None,
        max_seqlen:     Optional[int] = None,
    ) -> torch.Tensor:
        hidden_states = hidden_states + self.attention(
            self.ln_1(hidden_states),
            attention_mask=attention_mask, causal_mask=causal_mask,
            pos_xyz=pos_xyz, sdpa_mask=sdpa_mask, is_causal=is_causal,
            cu_seqlens=cu_seqlens, max_seqlen=max_seqlen,
        )
        hidden_states = hidden_states + self.ff(self.ln_2(hidden_states))
        return hidden_states

    def forward_with_cache(
        self,
        hidden_states:     torch.Tensor,
        pos_xyz:           torch.Tensor,
        attention_mask:    Optional[torch.Tensor] = None,
        causal_mask:       Optional[torch.Tensor] = None,
        past_key_value:    Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        cache_position:    Optional[torch.Tensor] = None,
        decode_block_mask: Optional[object] = None,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        attn_input = self.ln_1(hidden_states)
        if past_key_value is not None:
            # Decode step — no KV returned
            attn_out      = self.attention.forward_with_cache(
                attn_input, pos_xyz=pos_xyz,
                attention_mask=attention_mask, causal_mask=causal_mask,
                past_key_value=past_key_value, cache_position=cache_position,
                decode_block_mask=decode_block_mask,
            )
            present_kv = None
        else:
            # Prompt step — capture new KV tensors
            attn_out, present_kv = self.attention.forward_with_cache(
                attn_input, pos_xyz=pos_xyz,
                attention_mask=attention_mask, causal_mask=causal_mask,
            )

        hidden_states = hidden_states + attn_out
        hidden_states = hidden_states + self.ff(self.ln_2(hidden_states))

        # Zero-out tokens that are masked (decode only)
        if attention_mask is not None:
            seq_len = hidden_states.size(1)
            if cache_position is not None:
                positions = (
                    cache_position.view(1, -1)
                    + torch.arange(seq_len, device=cache_position.device).view(1, -1)
                ).clamp(max=attention_mask.size(1) - 1).expand(attention_mask.size(0), -1)
            else:
                positions = torch.arange(
                    attention_mask.size(1) - seq_len, attention_mask.size(1),
                    device=hidden_states.device,
                ).view(1, -1).expand(attention_mask.size(0), -1)
            token_mask = attention_mask.gather(1, positions)
            hidden_states = hidden_states * token_mask.unsqueeze(-1)

        if past_key_value is not None:
            return hidden_states
        return hidden_states, present_kv


# ── Main Model ─────────────────────────────────────────────────────────────────

class ARCTransformer(nn.Module):
    """Autoregressive transformer for ARC puzzle solving."""

    def __init__(self, config: ARCTransformerConfig) -> None:
        super().__init__()
        self.config = config

        self.token_embedding    = nn.Embedding(config.vocab_size, config.d_model)
        self.example_embedding  = nn.Embedding(config.num_examples, config.d_model)
        self.dihedral_embedding = nn.Embedding(config.num_dihedrals, config.d_model)
        self.dropout            = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.norm   = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.apply(self._init_weights)

        self._decode_block_mask_cache: Dict[Tuple[str, int, int, int], object] = {}
        self._decode_block_size = 128

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _build_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)
        return mask[None, None, :, :]

    def _get_3d_positions(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """Compute 3D positions on CPU, then move to target device."""
        pos_cpu = get_3d_positions(input_ids.detach().cpu(), attention_mask.detach().cpu())
        return pos_cpu.to(device=input_ids.device, dtype=torch.long)

    def _get_decode_block_mask(
        self, cache_position: int, kv_len: int, device: torch.device
    ) -> Optional[object]:
        if create_block_mask is None or kv_len < 1:
            return None
        block_idx    = max(int(cache_position), 0) // self._decode_block_size
        device_index = -1 if device.index is None else int(device.index)
        cache_key    = (device.type, device_index, int(kv_len), block_idx)
        cached       = self._decode_block_mask_cache.get(cache_key)
        if cached is not None:
            return cached

        block_end = min(((block_idx + 1) * self._decode_block_size) - 1, kv_len - 1)

        def block_superset_mask(b, h, q_idx, kv_idx):
            del b, h, q_idx
            return kv_idx <= block_end

        kwargs: Dict[str, Any] = {
            "mask_mod": block_superset_mask, "B": None, "H": None,
            "Q_LEN": 1, "KV_LEN": int(kv_len),
            "device": device, "BLOCK_SIZE": self._decode_block_size,
        }
        try:
            block_mask = (torch.compile(create_block_mask) if hasattr(torch, "compile") else create_block_mask)(**kwargs)
        except TypeError:
            try:
                block_mask = create_block_mask(**kwargs)
            except TypeError:
                block_mask = create_block_mask(block_superset_mask, 1, 1, 1, int(kv_len), device=device, BLOCK_SIZE=self._decode_block_size)

        self._decode_block_mask_cache[cache_key] = block_mask
        return block_mask

    def build_decode_block_mask_for_step(
        self,
        attention_mask: Optional[torch.Tensor],
        cache_position: Optional[torch.Tensor],
        kv_len:         int,
        device:         torch.device,
    ) -> Optional[object]:
        if attention_mask is None or cache_position is None:
            return None
        if attention_mask.dim() != 2 or attention_mask.size(1) < kv_len:
            return None

        cache_pos  = int(cache_position.reshape(-1)[0].item())
        block_mask = self._get_decode_block_mask(cache_pos, kv_len=kv_len, device=device)
        if block_mask is None:
            return None

        key_mask  = attention_mask[:, :kv_len]
        q_offset  = cache_position.reshape(-1)[0]

        def decode_mask_mod(b, h, q_idx, kv_idx):
            del h
            return (q_idx + q_offset >= kv_idx) & key_mask[b, kv_idx]

        try:
            block_mask.mask_mod = decode_mask_mod
        except Exception:
            return None
        return block_mask

    # ── Embedding helper ──────────────────────────────────────────────────────

    def _compute_embeddings_batched(
        self,
        input_ids:    torch.Tensor,
        example_ids:  torch.Tensor,
        dihedral_ids: torch.Tensor,
    ) -> torch.Tensor:
        return (
            self.token_embedding(input_ids)
            + self.example_embedding(example_ids).unsqueeze(1)
            + self.dihedral_embedding(dihedral_ids).unsqueeze(1)
        )

    # ── Forward router ────────────────────────────────────────────────────────

    def forward(
        self,
        input_ids:          torch.Tensor,
        example_ids:        torch.Tensor,
        dihedral_ids:       torch.Tensor,
        attention_mask:     Optional[torch.Tensor] = None,
        sep_indices:        Optional[torch.Tensor] = None,
        compute_input_loss: bool = True,
        targets:            Optional[torch.Tensor] = None,
        positions_3d:       Optional[torch.Tensor] = None,
        cu_seqlens:         Optional[torch.Tensor] = None,
        max_seqlen:         Optional[int] = None,
    ) -> dict:
        if input_ids.dim() == 1:
            return self._forward_varlen(
                input_ids, example_ids, dihedral_ids,
                sep_indices, compute_input_loss, targets, positions_3d, cu_seqlens, max_seqlen,
            )
        if input_ids.dim() == 2:
            return self._forward_padded(
                input_ids, example_ids, dihedral_ids,
                attention_mask, sep_indices, compute_input_loss, targets, positions_3d,
            )
        raise ValueError("input_ids must have shape [B, S] or [total_tokens].")

    # ── Padded (batched) forward ───────────────────────────────────────────────

    def _forward_padded(
        self,
        input_ids:          torch.Tensor,
        example_ids:        torch.Tensor,
        dihedral_ids:       torch.Tensor,
        attention_mask:     Optional[torch.Tensor],
        sep_indices:        Optional[torch.Tensor],
        compute_input_loss: bool,
        targets:            Optional[torch.Tensor],
        positions_3d:       Optional[torch.Tensor],
    ) -> dict:
        batch_size, seq_len = input_ids.size()
        if seq_len > self.config.max_seq_len:
            raise ValueError(f"Sequence length {seq_len} exceeds model capacity ({self.config.max_seq_len}).")

        device = input_ids.device

        # ── Input validation ───────────────────────────────────────────────────
        if sep_indices is not None:
            sep_indices = sep_indices.to(device=device, dtype=torch.long)
            if sep_indices.dim() != 1 or sep_indices.size(0) != batch_size:
                raise ValueError("sep_indices must have shape [batch_size].")
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids, dtype=torch.bool, device=device)
        else:
            attention_mask = attention_mask.to(device=device, dtype=torch.bool)
        if positions_3d is not None and positions_3d.shape[:2] != input_ids.shape:
            raise ValueError("positions_3d must match [batch, seq_len] of input_ids.")
        example_ids  = example_ids.to(device=device, dtype=torch.long)
        dihedral_ids = dihedral_ids.to(device=device, dtype=torch.long)
        if example_ids.dim() != 1 or example_ids.size(0) != batch_size:
            raise ValueError("example_ids must have shape [batch_size].")
        if dihedral_ids.dim() != 1 or dihedral_ids.size(0) != batch_size:
            raise ValueError("dihedral_ids must have shape [batch_size].")

        if targets is None:
            targets = input_ids

        # ── Embedding + encoding ───────────────────────────────────────────────
        hidden_states = self.dropout(self._compute_embeddings_batched(input_ids, example_ids, dihedral_ids))

        pos_xyz = (
            positions_3d.to(device=device, dtype=torch.long)
            if positions_3d is not None
            else self._get_3d_positions(input_ids, attention_mask)
        )

        for block in self.blocks:
            hidden_states = block(
                hidden_states,
                attention_mask=None, causal_mask=None,
                pos_xyz=pos_xyz, sdpa_mask=None, is_causal=True,
            )
            hidden_states = hidden_states * attention_mask.unsqueeze(-1)

        logits = self.lm_head(self.norm(hidden_states))

        # ── Loss computation ───────────────────────────────────────────────────
        loss = input_loss = output_loss = num_output_tokens = None

        if targets is not None:
            shift_logits  = logits[:, :-1, :].contiguous()
            shift_targets = targets[:, 1:].contiguous()
            shift_mask    = attention_mask[:, 1:].contiguous()
            shift_targets = shift_targets.masked_fill(~shift_mask, PAD_LABEL)

            raw_losses = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_targets.view(-1),
                ignore_index=PAD_LABEL, reduction="none",
            ).view(batch_size, -1)

            valid_mask  = shift_targets != PAD_LABEL
            total_valid = valid_mask.sum()
            loss        = raw_losses.sum() / total_valid.clamp(min=1)

            shift_len = shift_targets.size(1)
            if sep_indices is not None:
                sep_pos        = sep_indices.clamp(0, shift_len).unsqueeze(1)
                positions      = torch.arange(shift_len, device=device).unsqueeze(0)
                is_output_phase = positions >= sep_pos
            else:
                shift_input_ids = input_ids[:, :-1]
                is_output_phase = (shift_input_ids == SEP_ID).cumsum(dim=1) >= 1

            valid_output      = valid_mask & is_output_phase
            num_output_tokens = valid_output.sum()

            if compute_input_loss:
                valid_input = valid_mask & (~is_output_phase)
                input_loss  = (raw_losses * valid_input).sum() / valid_input.sum().clamp(min=1)
            output_loss = (raw_losses * valid_output).sum() / num_output_tokens.clamp(min=1)

        return {
            "logits":            logits,
            "loss":              loss,
            "input_loss":        input_loss,
            "output_loss":       output_loss,
            "num_output_tokens": num_output_tokens if targets is not None else None,
        }

    # ── Packed (varlen) forward ────────────────────────────────────────────────

    def _forward_varlen(
        self,
        input_ids:          torch.Tensor,
        example_ids:        torch.Tensor,
        dihedral_ids:       torch.Tensor,
        sep_indices:        Optional[torch.Tensor],
        compute_input_loss: bool,
        targets:            Optional[torch.Tensor],
        positions_3d:       Optional[torch.Tensor],
        cu_seqlens:         Optional[torch.Tensor],
        max_seqlen:         Optional[int],
    ) -> dict:
        if cu_seqlens is None:
            raise ValueError("cu_seqlens is required for packed varlen inputs.")
        if positions_3d is None:
            raise ValueError("positions_3d is required for packed varlen inputs.")
        if example_ids.dim() != 1:
            raise ValueError("example_ids must have shape [batch_size] for varlen inputs.")
        if dihedral_ids.dim() != 1:
            raise ValueError("dihedral_ids must have shape [batch_size] for varlen inputs.")

        device       = input_ids.device
        total_tokens = int(input_ids.size(0))
        batch_size   = int(example_ids.size(0))

        if int(dihedral_ids.size(0)) != batch_size:
            raise ValueError("dihedral_ids must match example_ids shape [batch_size].")
        if positions_3d.shape != (total_tokens, 3):
            raise ValueError("positions_3d must match packed shape [total_tokens, 3].")

        cu_seqlens = cu_seqlens.to(device=device, dtype=torch.int32)
        if cu_seqlens.dim() != 1 or cu_seqlens.size(0) != batch_size + 1:
            raise ValueError("cu_seqlens must have shape [batch_size + 1].")
        if int(cu_seqlens[0].item()) != 0:
            raise ValueError("cu_seqlens must start at 0.")
        if int(cu_seqlens[-1].item()) != total_tokens:
            raise ValueError("cu_seqlens[-1] must equal the packed token count in input_ids.")

        seq_lengths = (cu_seqlens[1:] - cu_seqlens[:-1]).to(dtype=torch.long)
        if torch.any(seq_lengths <= 0):
            raise ValueError("All sequences must have positive length in cu_seqlens.")

        max_seqlen_resolved = int(seq_lengths.max().item()) if max_seqlen is None else int(max_seqlen)
        if max_seqlen_resolved > self.config.max_seq_len:
            raise ValueError(f"Sequence length {max_seqlen_resolved} exceeds model capacity ({self.config.max_seq_len}).")

        if sep_indices is not None:
            sep_indices = sep_indices.to(device=device, dtype=torch.long)
            if sep_indices.dim() != 1 or sep_indices.size(0) != batch_size:
                raise ValueError("sep_indices must have shape [batch_size].")

        # Build targets (shift-by-one, mask sequence end tokens)
        if targets is not None:
            if targets.dim() != 1 or targets.size(0) != total_tokens:
                raise ValueError("targets must have shape [total_tokens] for varlen inputs.")
            targets = targets.to(device=device, dtype=torch.long)
        else:
            targets                  = torch.roll(input_ids, shifts=-1).clone()
            sequence_ends            = cu_seqlens[1:].to(dtype=torch.long) - 1
            targets[sequence_ends]   = PAD_LABEL

        example_ids  = example_ids.to(device=device, dtype=torch.long)
        dihedral_ids = dihedral_ids.to(device=device, dtype=torch.long)

        # ── Embedding ──────────────────────────────────────────────────────────
        seq_embeds    = self.example_embedding(example_ids) + self.dihedral_embedding(dihedral_ids)
        token_seq_emb = torch.repeat_interleave(seq_embeds, seq_lengths, dim=0)
        hidden_states = self.dropout(self.token_embedding(input_ids) + token_seq_emb)

        pos_xyz = positions_3d.to(device=device, dtype=torch.long)

        for block in self.blocks:
            hidden_states = block(
                hidden_states,
                attention_mask=None, causal_mask=None,
                pos_xyz=pos_xyz, sdpa_mask=None, is_causal=True,
                cu_seqlens=cu_seqlens, max_seqlen=max_seqlen_resolved,
            )

        logits = self.lm_head(self.norm(hidden_states))

        # ── Loss ──────────────────────────────────────────────────────────────
        raw_losses = F.cross_entropy(logits, targets, ignore_index=PAD_LABEL, reduction="none")
        valid_mask = targets != PAD_LABEL
        loss       = raw_losses.sum() / valid_mask.sum().clamp(min=1)

        seq_ids = torch.repeat_interleave(
            torch.arange(batch_size, device=device, dtype=torch.long), seq_lengths,
        )
        local_pos = (
            torch.arange(total_tokens, device=device, dtype=torch.long)
            - torch.repeat_interleave(cu_seqlens[:-1].to(dtype=torch.long), seq_lengths)
        )

        if sep_indices is not None:
            is_output_phase = local_pos >= sep_indices[seq_ids]
        else:
            is_sep          = (input_ids == SEP_ID).long()
            all_sep_cumsum  = is_sep.cumsum(0)
            starts          = cu_seqlens[:-1].long()
            pre_seq         = torch.zeros(batch_size, device=device, dtype=torch.long)
            if batch_size > 1:
                pre_seq[1:] = all_sep_cumsum[starts[1:] - 1]
            is_output_phase = (all_sep_cumsum - torch.repeat_interleave(pre_seq, seq_lengths)) >= 1

        valid_output      = valid_mask & is_output_phase
        num_output_tokens = valid_output.sum()

        input_loss = None
        if compute_input_loss:
            valid_input = valid_mask & (~is_output_phase)
            input_loss  = (raw_losses * valid_input).sum() / valid_input.sum().clamp(min=1)
        output_loss = (raw_losses * valid_output).sum() / num_output_tokens.clamp(min=1)

        return {
            "logits":            logits,
            "loss":              loss,
            "input_loss":        input_loss,
            "output_loss":       output_loss,
            "num_output_tokens": num_output_tokens,
        }

    # ── Generation (KV-cache) forward ─────────────────────────────────────────

    def forward_generate(
        self,
        input_ids:         torch.Tensor,
        example_ids:       torch.Tensor,
        dihedral_ids:      torch.Tensor,
        past_key_values:   Optional[Tuple[Tuple[torch.Tensor, torch.Tensor], ...]] = None,
        positions_3d:      Optional[torch.Tensor] = None,
        attention_mask:    Optional[torch.Tensor] = None,
        cache_position:    Optional[torch.Tensor] = None,
        example_embeds:    Optional[torch.Tensor] = None,
        decode_block_mask: Optional[object] = None,
    ) -> dict:
        """Autoregressive generation with a KV cache.

        Prompt pass  (past_key_values=None): full forward; returns per-layer KV tensors.
        Decode step  (past_key_values set):  single-token forward; updates cache in place.
        """
        batch_size, seq_len = input_ids.size()
        if seq_len > self.config.max_seq_len:
            raise ValueError(f"Sequence length {seq_len} exceeds model capacity ({self.config.max_seq_len}).")
        if positions_3d is not None and positions_3d.shape[:2] != input_ids.shape:
            raise ValueError("positions_3d must match [batch, seq_len] of input_ids.")

        device       = input_ids.device
        example_ids  = example_ids.to(device=device, dtype=torch.long)
        dihedral_ids = dihedral_ids.to(device=device, dtype=torch.long)

        if example_ids.dim() != 1 or example_ids.size(0) != batch_size:
            raise ValueError("example_ids must have shape [batch_size].")
        if dihedral_ids.dim() != 1 or dihedral_ids.size(0) != batch_size:
            raise ValueError("dihedral_ids must have shape [batch_size].")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device=device, dtype=torch.bool)

        if example_embeds is not None:
            if example_embeds.shape[0] != batch_size:
                raise ValueError("example_embeds batch size must match input_ids.")
        else:
            example_embeds = self.example_embedding(example_ids)

        hidden_states = (
            self.token_embedding(input_ids)
            + example_embeds.unsqueeze(1)
            + self.dihedral_embedding(dihedral_ids).unsqueeze(1)
        )

        pos_xyz = positions_3d.to(device=device, dtype=torch.long) if positions_3d is not None else None

        if past_key_values is None:
            # ── Prompt pass ──────────────────────────────────────────────────
            if attention_mask is None:
                attention_mask = torch.ones_like(input_ids, dtype=torch.bool, device=device)
            if pos_xyz is None:
                pos_xyz = self._get_3d_positions(input_ids, attention_mask)

            causal_mask = self._build_causal_mask(seq_len, device)
            new_kvs: List[Tuple[torch.Tensor, torch.Tensor]] = []
            for block in self.blocks:
                hidden_states, kv = block.forward_with_cache(
                    hidden_states, pos_xyz,
                    attention_mask=attention_mask, causal_mask=causal_mask,
                )
                new_kvs.append(kv)

            logits = self.lm_head(self.norm(hidden_states[:, -1:, :]))
            return {"logits": logits, "past_key_values": tuple(new_kvs)}

        # ── Decode step ───────────────────────────────────────────────────────
        if pos_xyz is None:
            raise ValueError("positions_3d must be provided when using past_key_values.")

        for i, block in enumerate(self.blocks):
            hidden_states = block.forward_with_cache(
                hidden_states, pos_xyz,
                attention_mask=attention_mask,
                past_key_value=past_key_values[i],
                cache_position=cache_position,
                decode_block_mask=decode_block_mask,
            )

        logits = self.lm_head(self.norm(hidden_states))
        return {"logits": logits}


## Training Loop & NorMuon Optimizer


In [5]:
# ── Newton-Schulz orthogonalisation (used by NorMuon) ────────────────────────

def _zeropower_via_newtonschulz5(G, steps=5):
    """Compute the matrix zeroth-power (orthogonalisation) via Newton-Schulz iteration."""
    assert G.ndim >= 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    if G.size(-2) > G.size(-1):
        X = X.mT
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(-2) > G.size(-1):
        X = X.mT
    return X


def _normuon_update(grad, momentum, second_momentum, beta=0.95, beta2=0.95, ns_steps=5, nesterov=True):
    """One NorMuon update: Nesterov momentum + Newton-Schulz orthogonalisation + RMS normalisation."""
    momentum.lerp_(grad, 1 - beta)
    update = torch.lerp(grad, momentum, beta) if nesterov else momentum

    original_shape = None
    if update.ndim == 4:
        original_shape = update.shape
        update = update.reshape(update.size(0), -1)

    update = _zeropower_via_newtonschulz5(update, steps=ns_steps)
    if original_shape is not None:
        update = update.reshape(original_shape)

    v_norm = update.norm(dim=(-2, -1), keepdim=True)
    v_mean = update.square().mean(dim=-1, keepdim=True)
    second_momentum.lerp_(v_mean.to(second_momentum.dtype), 1 - beta2)
    step_size = 1 / second_momentum.sqrt().add_(1e-10)
    update.mul_(step_size)

    v_norm_new = update.norm(dim=(-2, -1), keepdim=True)
    update.mul_(v_norm / (v_norm_new.add_(1e-10)))
    update *= max(1, grad.size(-2) / grad.size(-1)) ** 0.5
    return update


def _adam_update(grad, buf1, buf2, step, betas, eps):
    """One Adam update: EMA of gradient and squared gradient, bias-corrected."""
    buf1.lerp_(grad, 1 - betas[0])
    buf2.lerp_(grad.square(), 1 - betas[1])
    buf1c = buf1 / (1 - betas[0] ** step)
    buf2c = buf2 / (1 - betas[1] ** step)
    return buf1c / (buf2c.sqrt() + eps)


# ── NorMuon Optimizer ─────────────────────────────────────────────────────────

class NorMuonOptimizer(torch.optim.Optimizer):
    """NorMuon optimizer: Muon (Newton-Schulz) for linear weights, AdamW for everything else."""

    def __init__(self, param_groups):
        for group in param_groups:
            assert "use_muon" in group
            if group["use_muon"]:
                group["lr"]           = group.get("lr", 0.02)
                group["momentum"]     = group.get("momentum", 0.95)
                group["beta2"]        = group.get("beta2", 0.95)
                group["weight_decay"] = group.get("weight_decay", 0.1)
                assert set(group.keys()) == {"params", "lr", "momentum", "beta2", "weight_decay", "use_muon"}
            else:
                group["lr"]           = group.get("lr", 3e-4)
                group["betas"]        = group.get("betas", (0.9, 0.95))
                group["eps"]          = group.get("eps", 1e-10)
                group["weight_decay"] = group.get("weight_decay", 0.1)
                assert set(group.keys()) == {"params", "lr", "betas", "eps", "weight_decay", "use_muon"}
        super().__init__(param_groups, dict())

        self._normuon_update_fn = _normuon_update
        self._adam_update_fn    = _adam_update
        self._compiled_kernels  = False
        self._zero_scalars: Dict[Tuple[torch.device, torch.dtype], torch.Tensor] = {}

        disable_compile = str(os.environ.get("DISABLE_OPTIMIZER_COMPILE", "0")).lower() in {"1", "true", "yes", "on"}
        has_cuda_params = any(p.is_cuda for group in param_groups for p in group["params"])
        gpu_name        = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
        compile_ok = (
            has_cuda_params and hasattr(torch, "compile")
            and not disable_compile and "T4" not in gpu_name
        )
        if compile_ok:
            try:
                self._normuon_update_fn = torch.compile(_normuon_update, mode="reduce-overhead", fullgraph=False, dynamic=True)
                self._adam_update_fn    = torch.compile(_adam_update,    mode="reduce-overhead", fullgraph=False, dynamic=True)
                self._compiled_kernels  = True
                print("Compiled NorMuon optimizer update kernels.")
            except Exception as exc:
                print(f"Skipping NorMuon optimizer kernel compile ({exc}).")
        else:
            print(f"Skipping NorMuon kernel compile (GPU: {gpu_name or 'cpu'}).")

    def _get_zero_grad(self, p: torch.Tensor) -> torch.Tensor:
        key  = (p.device, p.dtype)
        zero = self._zero_scalars.get(key)
        if zero is None:
            zero = torch.zeros((), device=p.device, dtype=p.dtype)
            self._zero_scalars[key] = zero
        return zero.expand_as(p)

    def _call_normuon_update(self, grad, momentum, second_momentum, beta, beta2):
        if not self._compiled_kernels:
            return _normuon_update(grad, momentum, second_momentum, beta=beta, beta2=beta2)
        try:
            return self._normuon_update_fn(grad, momentum, second_momentum, beta=beta, beta2=beta2)
        except Exception as exc:
            print(f"Compiled NorMuon kernels failed ({exc}); falling back to eager.")
            self._compiled_kernels  = False
            self._normuon_update_fn = _normuon_update
            self._adam_update_fn    = _adam_update
            return _normuon_update(grad, momentum, second_momentum, beta=beta, beta2=beta2)

    def _call_adam_update(self, grad, buf1, buf2, step, betas, eps):
        if not self._compiled_kernels:
            return _adam_update(grad, buf1, buf2, step, betas, eps)
        try:
            return self._adam_update_fn(grad, buf1, buf2, step, betas, eps)
        except Exception as exc:
            print(f"Compiled NorMuon kernels failed ({exc}); falling back to eager.")
            self._compiled_kernels  = False
            self._normuon_update_fn = _normuon_update
            self._adam_update_fn    = _adam_update
            return _adam_update(grad, buf1, buf2, step, betas, eps)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            if group["use_muon"]:
                self._muon_step(group)
            else:
                self._adam_step(group)
        return loss

    def _muon_step(self, group):
        lr           = float(group["lr"])
        beta         = float(group["momentum"])
        beta2        = float(group["beta2"])
        weight_decay = float(group["weight_decay"])

        # Bucket params by shape+dtype+device for batched updates
        buckets: Dict = {}
        for p in group["params"]:
            grad     = p.grad
            had_grad = grad is not None
            if grad is None:
                grad = self._get_zero_grad(p)
            state = self.state[p]
            if len(state) == 0:
                state["momentum_buffer"]        = torch.zeros_like(p)
                state["second_momentum_buffer"] = torch.zeros_like(p[..., 0:1])
            key = (tuple(p.shape), p.dtype, p.device)
            buckets.setdefault(key, []).append((p, grad, state["momentum_buffer"], state["second_momentum_buffer"], had_grad))

        params_to_update: List[torch.Tensor] = []
        updates:          List[torch.Tensor] = []
        params_with_grad: List[torch.Tensor] = []

        for bucket in buckets.values():
            if len(bucket) == 1:
                p, grad, mom, sec_mom, had_grad = bucket[0]
                update = self._call_normuon_update(grad, mom, sec_mom, beta, beta2)
                params_to_update.append(p)
                updates.append(update.reshape_as(p))
                if weight_decay and had_grad:
                    params_with_grad.append(p)
            else:
                grad_batch    = torch.stack([item[1] for item in bucket])
                mom_batch     = torch.stack([item[2] for item in bucket])
                sec_mom_batch = torch.stack([item[3] for item in bucket])
                update_batch  = self._call_normuon_update(grad_batch, mom_batch, sec_mom_batch, beta, beta2)
                for idx, (p, _grad, mom, sec_mom, had_grad) in enumerate(bucket):
                    mom.copy_(mom_batch[idx])
                    sec_mom.copy_(sec_mom_batch[idx])
                    params_to_update.append(p)
                    updates.append(update_batch[idx].reshape_as(p))
                    if weight_decay and had_grad:
                        params_with_grad.append(p)

        if weight_decay and params_with_grad:
            torch._foreach_mul_(params_with_grad, 1 - lr * weight_decay)
        if params_to_update:
            torch._foreach_add_(params_to_update, updates, alpha=-lr)

    def _adam_step(self, group):
        lr           = float(group["lr"])
        betas        = group["betas"]
        beta1, beta2 = float(betas[0]), float(betas[1])
        eps          = float(group["eps"])
        weight_decay = float(group["weight_decay"])

        params, grads, exp_avgs, exp_avg_sqs, steps = [], [], [], [], []
        params_with_grad: List[torch.Tensor] = []

        for p in group["params"]:
            grad     = p.grad
            had_grad = grad is not None
            if grad is None:
                grad = self._get_zero_grad(p)
            state = self.state[p]
            if len(state) == 0:
                state["exp_avg"]    = torch.zeros_like(p)
                state["exp_avg_sq"] = torch.zeros_like(p)
                state["step"]       = 0
            state["step"] += 1
            params.append(p)
            grads.append(grad)
            exp_avgs.append(state["exp_avg"])
            exp_avg_sqs.append(state["exp_avg_sq"])
            steps.append(int(state["step"]))
            if weight_decay and had_grad:
                params_with_grad.append(p)

        if not params:
            return

        all_same_step = all(s == steps[0] for s in steps)
        if all_same_step:
            torch._foreach_lerp_(exp_avgs, grads, 1 - beta1)
            torch._foreach_mul_(exp_avg_sqs, beta2)
            torch._foreach_addcmul_(exp_avg_sqs, grads, grads, value=1 - beta2)
            step_val         = steps[0]
            bias_correction1 = 1 - beta1 ** step_val
            bias_correction2 = 1 - beta2 ** step_val
            updates = torch._foreach_div(exp_avgs, bias_correction1)
            denoms  = torch._foreach_sqrt(exp_avg_sqs)
            torch._foreach_div_(denoms, math.sqrt(bias_correction2))
            torch._foreach_add_(denoms, eps)
            updates = torch._foreach_div(updates, denoms)
        else:
            updates = [
                self._call_adam_update(g, ea, eas, s, betas, eps)
                for g, ea, eas, s in zip(grads, exp_avgs, exp_avg_sqs, steps)
            ]

        if weight_decay and params_with_grad:
            torch._foreach_mul_(params_with_grad, 1 - lr * weight_decay)
        torch._foreach_add_(params, updates, alpha=-lr)


# ── Logging helper ────────────────────────────────────────────────────────────

def _emit_log(message: str, log_location: str, log_handle: Optional[TextIO]) -> None:
    """Write a log message to terminal, file, or both."""
    if log_location in ("terminal", "both"):
        print(message)
    if log_location in ("file", "both") and log_handle is not None:
        log_handle.write(message + "\n")
        log_handle.flush()


# ── Training epoch ────────────────────────────────────────────────────────────

def train_epoch(
    model:                        ARCTransformer,
    dataloader:                   torch.utils.data.DataLoader,
    optimizer:                    torch.optim.Optimizer,
    device:                       torch.device,
    grad_clip:                    float,
    gradient_accumulation_steps:  int = 1,
    start_step:                   int = 0,
    scheduler:                    Optional[torch.optim.lr_scheduler.LRScheduler] = None,
    epoch:                        Optional[int] = None,
    steps_per_epoch:              Optional[int] = None,
    train_log_mode:               str = "10_steps",
    log_location:                 str = "both",
    log_handle:                   Optional[TextIO] = None,
) -> int:
    model.train()
    step    = start_step
    use_amp = device.type == "cuda"

    if steps_per_epoch is not None and steps_per_epoch <= 0:
        steps_per_epoch = None
    if steps_per_epoch is None:
        steps_per_epoch = len(dataloader)

    accum_steps  = max(1, int(gradient_accumulation_steps or 1))
    accum_index  = 0
    accum_target = accum_steps
    opt_step     = 0

    # Rolling and epoch loss accumulators (on device to avoid host syncs)
    window_loss  = window_inp  = window_out  = window_n  = 0
    epoch_loss   = epoch_inp   = epoch_out   = epoch_n   = 0
    win_loss_t   = torch.zeros((), device=device, dtype=torch.float32)
    win_inp_t    = torch.zeros((), device=device, dtype=torch.float32)
    win_out_t    = torch.zeros((), device=device, dtype=torch.float32)
    ep_loss_t    = torch.zeros((), device=device, dtype=torch.float32)
    ep_inp_t     = torch.zeros((), device=device, dtype=torch.float32)
    ep_out_t     = torch.zeros((), device=device, dtype=torch.float32)
    win_count    = ep_count = 0

    def maybe_log_window(force: bool = False) -> None:
        nonlocal win_count
        should_log = (
            train_log_mode == "step"
            or (train_log_mode == "10_steps" and ((opt_step % 10 == 0) or force))
        )
        if not should_log or win_count == 0:
            return
        avg_loss = (win_loss_t / win_count).item()
        avg_inp  = (win_inp_t  / win_count).item()
        avg_out  = (win_out_t  / win_count).item()
        current_lr = (scheduler.get_last_lr()[0] if scheduler else optimizer.param_groups[0]["lr"])
        prefix = f"epoch={epoch + 1} step={opt_step}" if epoch is not None else f"step={opt_step}"
        _emit_log(
            f"{prefix} lr={current_lr:.2e} losses: avg={avg_loss:.4f} inp={avg_inp:.4f} out={avg_out:.4f}",
            log_location, log_handle,
        )
        win_loss_t.zero_(); win_inp_t.zero_(); win_out_t.zero_()
        win_count = 0

    last_batch_idx = None
    for batch_idx, batch in enumerate(dataloader):
        last_batch_idx = batch_idx
        step += 1

        input_ids  = batch["input_ids"].to(device)
        sep_indices = batch.get("sep_indices")
        sep_indices = sep_indices.to(device) if sep_indices is not None else None

        cu_seqlens_cpu = batch.get("cu_seqlens")
        if cu_seqlens_cpu is not None:
            cu_seqlens = cu_seqlens_cpu.to(device=device, dtype=torch.int32)
            max_seqlen_raw = batch.get("max_seqlen")
            if max_seqlen_raw is None:
                raise ValueError("Packed batches must include max_seqlen.")
            max_seqlen     = int(max_seqlen_raw.item()) if torch.is_tensor(max_seqlen_raw) else int(max_seqlen_raw)
            attention_mask = None
        else:
            cu_seqlens     = None
            max_seqlen     = None
            has_padding    = bool(batch.get("has_padding", True))
            attention_mask = batch["attention_mask"].to(device) if has_padding else None

        example_ids  = batch["example_ids"].to(device)
        dihedral_ids = batch["dihedral_ids"].to(device)
        positions_3d = batch["positions_3d"].to(device)

        if accum_index == 0:
            optimizer.zero_grad(set_to_none=True)
            if steps_per_epoch is not None:
                remaining    = steps_per_epoch - batch_idx
                accum_target = min(accum_steps, remaining)
            else:
                accum_target = accum_steps

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp):
            outputs  = model(
                input_ids, example_ids, dihedral_ids,
                attention_mask=attention_mask, sep_indices=sep_indices,
                compute_input_loss=False, positions_3d=positions_3d,
                cu_seqlens=cu_seqlens, max_seqlen=max_seqlen,
            )
            loss     = outputs["output_loss"]
            inp_loss = outputs.get("input_loss")
            out_loss = outputs.get("output_loss")

        win_loss_t += loss.detach().float()
        ep_loss_t  += loss.detach().float()
        if inp_loss is not None:
            win_inp_t += inp_loss.detach().float()
            ep_inp_t  += inp_loss.detach().float()
        if out_loss is not None:
            win_out_t += out_loss.detach().float()
            ep_out_t  += out_loss.detach().float()
        win_count += 1
        ep_count  += 1

        (loss / accum_target).backward()
        accum_index += 1

        if accum_index >= accum_target:
            if grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            if scheduler is not None:
                if epoch is None or steps_per_epoch is None:
                    scheduler.step()
                else:
                    epoch_progress = float(epoch) + (batch_idx + 1) / float(steps_per_epoch)
                    scheduler.last_epoch = int(epoch_progress) - 1
                    scheduler.step()
            accum_index = 0
            opt_step   += 1
            maybe_log_window()

    # Flush any partial accumulation at end of epoch
    if accum_index > 0:
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        if scheduler is not None:
            if epoch is None or steps_per_epoch is None or last_batch_idx is None:
                scheduler.step()
            else:
                epoch_progress = float(epoch) + (last_batch_idx + 1) / float(steps_per_epoch)
                scheduler.last_epoch = int(epoch_progress) - 1
                scheduler.step()
        opt_step += 1
        maybe_log_window()

    if train_log_mode == "10_steps":
        maybe_log_window(force=True)
    elif train_log_mode == "epoch" and ep_count > 0:
        current_lr = scheduler.get_last_lr()[0] if scheduler else optimizer.param_groups[0]["lr"]
        prefix     = f"epoch={epoch + 1}" if epoch is not None else "epoch"
        _emit_log(
            f"{prefix} lr={current_lr:.2e} losses: avg={(ep_loss_t/ep_count).item():.4f} "
            f"inp={(ep_inp_t/ep_count).item():.4f} out={(ep_out_t/ep_count).item():.4f}",
            log_location, log_handle,
        )
    return step


# ── Validation epoch ──────────────────────────────────────────────────────────

@torch.no_grad()
def val_epoch(
    model:      ARCTransformer,
    dataloader: torch.utils.data.DataLoader,
    device:     torch.device,
) -> float:
    """Compute token-weighted output loss over the validation set."""
    model.eval()
    use_amp     = device.type == "cuda"
    total_loss  = torch.zeros((), device=device, dtype=torch.float32)
    total_toks  = torch.zeros((), device=device, dtype=torch.float32)

    for batch in dataloader:
        input_ids    = batch["input_ids"].to(device)
        sep_indices  = batch.get("sep_indices")
        sep_indices  = sep_indices.to(device) if sep_indices is not None else None
        cu_seqlens_cpu = batch.get("cu_seqlens")
        if cu_seqlens_cpu is not None:
            cu_seqlens = cu_seqlens_cpu.to(device=device, dtype=torch.int32)
            max_seqlen_raw = batch.get("max_seqlen")
            if max_seqlen_raw is None:
                raise ValueError("Packed batches must include max_seqlen.")
            max_seqlen     = int(max_seqlen_raw.item()) if torch.is_tensor(max_seqlen_raw) else int(max_seqlen_raw)
            attention_mask = None
        else:
            cu_seqlens     = None
            max_seqlen     = None
            has_padding    = bool(batch.get("has_padding", True))
            attention_mask = batch["attention_mask"].to(device) if has_padding else None

        example_ids  = batch["example_ids"].to(device)
        dihedral_ids = batch["dihedral_ids"].to(device)
        positions_3d = batch["positions_3d"].to(device)

        if not any(batch["has_output"]):
            continue

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp):
            outputs = model(
                input_ids, example_ids, dihedral_ids,
                attention_mask=attention_mask, sep_indices=sep_indices,
                compute_input_loss=False, positions_3d=positions_3d,
                cu_seqlens=cu_seqlens, max_seqlen=max_seqlen,
            )

        out_loss   = outputs.get("output_loss")
        num_tokens = outputs.get("num_output_tokens")

        if out_loss is not None and num_tokens is not None:
            n = num_tokens.detach().to(dtype=torch.float32)
            total_loss += out_loss.detach().float() * n
            total_toks += n

    return (total_loss / total_toks.clamp_min(1.0)).item()


# ── Parameter group helpers ───────────────────────────────────────────────────

def _is_norm_module(module: nn.Module) -> bool:
    return isinstance(module, (nn.LayerNorm, nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.GroupNorm, RMSNorm))


def _is_positional_embedding_param(name: str) -> bool:
    lower = name.lower()
    return "pos_embed" in lower or "position_embedding" in lower or "positional_embedding" in lower


def _is_attention_param(name: str) -> bool:
    return "attention" in name.split(".")


def _is_muon_candidate(module: nn.Module, name: str, param: nn.Parameter) -> bool:
    return (
        isinstance(module, nn.Linear)
        and not name.startswith("lm_head.")
        and name.endswith(".weight")
        and param.ndim == 2
    )


def _normuon_supported(device: torch.device) -> Tuple[bool, str]:
    if device.type != "cuda":
        return False, "NorMuon requires CUDA."
    bf16_ok = getattr(torch.cuda, "is_bf16_supported", None)
    if callable(bf16_ok) and not bf16_ok():
        return False, "NorMuon requires CUDA bfloat16 support"
    return True, ""


def _collect_param_groups(model: nn.Module, *, include_muon: bool) -> Dict[str, list]:
    groups: Dict[str, list] = {
        "decay": [], "attention": [], "token_embed": [], "task_embed": [], "no_decay": [],
        "muon": [], "muon_attention": [],
    }
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        module_name = name.rsplit(".", 1)[0] if "." in name else ""
        module      = model.get_submodule(module_name) if module_name else model
        is_attn     = _is_attention_param(name)

        if include_muon and _is_muon_candidate(module, name, param):
            groups["muon_attention" if is_attn else "muon"].append(param)
            continue
        if name.endswith(".bias"):
            groups["no_decay"].append(param); continue
        if _is_norm_module(module) or _is_positional_embedding_param(name):
            groups["no_decay"].append(param); continue
        if isinstance(module, nn.Embedding):
            if name.startswith("token_embedding."):
                groups["token_embed"].append(param)
            elif name.startswith(("example_embedding.", "dihedral_embedding.")):
                groups["task_embed"].append(param)
            else:
                groups["no_decay"].append(param)
            continue
        if is_attn:
            groups["attention"].append(param); continue
        if isinstance(module, nn.Linear):
            groups["decay"].append(param)
        else:
            groups["no_decay"].append(param)
    return groups


def _build_param_groups(
    model:                        nn.Module,
    weight_decay:                 float,
    attention_weight_decay:       float,
    token_embedding_weight_decay: float,
    task_embedding_weight_decay:  float,
) -> Tuple[Sequence[Dict[str, Any]], Sequence[Dict[str, Any]]]:
    """Split parameters into Muon groups (linear weights) and AdamW groups (everything else)."""
    groups = _collect_param_groups(model, include_muon=True)

    muon_groups = []
    if groups["muon"]:
        muon_groups.append({"params": groups["muon"], "weight_decay": weight_decay})
    if groups["muon_attention"]:
        muon_groups.append({"params": groups["muon_attention"], "weight_decay": attention_weight_decay})

    adamw_groups = []
    if groups["decay"]:
        adamw_groups.append({"params": groups["decay"], "weight_decay": weight_decay})
    if groups["attention"]:
        adamw_groups.append({"params": groups["attention"], "weight_decay": attention_weight_decay})
    if groups["token_embed"]:
        adamw_groups.append({"params": groups["token_embed"], "weight_decay": token_embedding_weight_decay})
    if groups["task_embed"]:
        adamw_groups.append({"params": groups["task_embed"], "weight_decay": task_embedding_weight_decay})
    if groups["no_decay"]:
        adamw_groups.append({"params": groups["no_decay"], "weight_decay": 0.0})

    return muon_groups, adamw_groups


def _move_optimizer_state(optimizer: torch.optim.Optimizer, device: torch.device) -> None:
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)


def _load_optimizer_state(optimizer: torch.optim.Optimizer, state_dict: Dict[str, Any], device: torch.device) -> bool:
    def _is_torch_sd(value):
        return isinstance(value, dict) and "param_groups" in value

    candidate = None
    if _is_torch_sd(state_dict):
        candidate = state_dict
    else:
        sub = state_dict.get(optimizer.__class__.__name__.lower())
        if _is_torch_sd(sub):
            candidate = sub

    if candidate is None:
        return False
    try:
        optimizer.load_state_dict(candidate)
    except (KeyError, ValueError, RuntimeError) as exc:
        print(f"Skipping optimizer state restore ({exc}).")
        return False
    _move_optimizer_state(optimizer, device)
    return True


def _optimizer_identity(optimizer: torch.optim.Optimizer) -> str:
    return optimizer.__class__.__name__.lower()


_OPTIMIZER_HPARAMS_IGNORE_KEYS = {"params", "fused", "foreach", "capturable", "differentiable"}


def _sanitize_optimizer_value(value: Any) -> Optional[Any]:
    if isinstance(value, bool):      return value
    if isinstance(value, numbers.Integral): return int(value)
    if isinstance(value, numbers.Number):   return float(value)
    if isinstance(value, str) or value is None: return value
    if isinstance(value, (list, tuple)):
        result = []
        for item in value:
            s = _sanitize_optimizer_value(item)
            if s is None and item is not None:
                return None
            result.append(s)
        return result
    if isinstance(value, dict):
        result = {}
        for k, v in value.items():
            s = _sanitize_optimizer_value(v)
            if s is None and v is not None:
                return None
            result[k] = s
        return result
    return None


def _param_groups_snapshot(param_groups: Sequence[Dict[str, Any]]) -> Sequence[Dict[str, Any]]:
    snapshot = []
    for group in param_groups:
        entry = {}
        for key, value in group.items():
            if key in _OPTIMIZER_HPARAMS_IGNORE_KEYS:
                continue
            s = _sanitize_optimizer_value(value)
            if s is None and value is not None:
                continue
            entry[key] = s
        snapshot.append(entry)
    return snapshot


def _checkpoint_optimizer_hparams(checkpoint: Dict[str, Any]) -> Optional[Any]:
    sd = checkpoint.get("optimizer_state")
    if isinstance(sd, dict) and "param_groups" in sd:
        return _param_groups_snapshot(sd.get("param_groups", []))
    return None


def _optimizer_values_match(left: Any, right: Any) -> bool:
    if isinstance(left, float) and isinstance(right, float):
        return math.isclose(left, right, rel_tol=1e-6, abs_tol=1e-8)
    if isinstance(left, (list, tuple)) and isinstance(right, (list, tuple)):
        return len(left) == len(right) and all(_optimizer_values_match(l, r) for l, r in zip(left, right))
    if isinstance(left, dict) and isinstance(right, dict):
        return left.keys() == right.keys() and all(_optimizer_values_match(left[k], right[k]) for k in left)
    return left == right


def _optimizer_hparams_changed(optimizer: torch.optim.Optimizer, checkpoint: Dict[str, Any]) -> bool:
    saved   = checkpoint.get("optimizer_hparams") or _checkpoint_optimizer_hparams(checkpoint)
    if saved is None:
        return False
    return not _optimizer_values_match(_param_groups_snapshot(optimizer.param_groups), saved)


def _apply_param_group_hparams(param_groups, hparams) -> None:
    if not isinstance(hparams, Sequence):
        return
    for group, desired in zip(param_groups, hparams):
        if isinstance(desired, dict):
            group.update(desired)


def _optimizer_switch_detected(optimizer: torch.optim.Optimizer, checkpoint: Dict[str, Any]) -> bool:
    ckpt_name = checkpoint.get("optimizer_name")
    if ckpt_name:
        return str(ckpt_name).lower() != _optimizer_identity(optimizer)
    sd = checkpoint.get("optimizer_state")
    if isinstance(sd, dict) and "muon" in sd and "adamw" in sd:
        return True
    return False


def _build_optimizer(
    args:                         argparse.Namespace,
    model:                        nn.Module,
    device:                       torch.device,
    attention_weight_decay:       float,
    token_embedding_weight_decay: float,
    task_embedding_weight_decay:  float,
) -> torch.optim.Optimizer:
    optimizer_name = str(getattr(args, "optimizer", "adamw")).lower()
    use_fused      = device.type == "cuda"

    muon_groups, adamw_groups = _build_param_groups(
        model, args.weight_decay, attention_weight_decay,
        token_embedding_weight_decay, task_embedding_weight_decay,
    )

    if optimizer_name != "normuon":
        return AdamW(list(muon_groups) + list(adamw_groups), lr=args.adamw_lr, fused=use_fused)

    supported, reason = _normuon_supported(device)
    if not supported:
        print(f"NorMuon unavailable ({reason}); falling back to AdamW.")
        return AdamW(list(muon_groups) + list(adamw_groups), lr=args.adamw_lr, fused=use_fused)
    if not muon_groups:
        print("NorMuon requested but no eligible linear weights found; using AdamW.")
        return AdamW(list(adamw_groups), lr=args.adamw_lr, fused=use_fused)

    normuon_lr       = float(getattr(args, "normuon_lr",       0.02))
    normuon_momentum = float(getattr(args, "normuon_momentum", 0.95))
    normuon_beta2    = float(getattr(args, "normuon_beta2",    0.95))
    adamw_lr         = float(getattr(args, "adamw_lr",         3e-4))

    normuon_param_groups = []
    for group in muon_groups:
        g = dict(group)
        g.update({"use_muon": True, "lr": normuon_lr, "momentum": normuon_momentum, "beta2": normuon_beta2})
        normuon_param_groups.append(g)
    for group in adamw_groups:
        g = dict(group)
        g.update({"use_muon": False, "lr": adamw_lr})
        normuon_param_groups.append(g)
    return NorMuonOptimizer(normuon_param_groups)


# ── Checkpoint I/O ────────────────────────────────────────────────────────────

def save_checkpoint(
    model:        ARCTransformer,
    dataset:      PuzzleDataset,
    data_path:    Path,
    save_path:    Optional[Path],
    optimizer:    Optional[torch.optim.Optimizer] = None,
    global_step:  Optional[int] = None,
    epoch:        Optional[int] = None,
    rng_state:    Optional[Dict[str, Any]] = None,
    scheduler:    Optional[torch.optim.lr_scheduler.LRScheduler] = None,
) -> None:
    if save_path is None:
        return
    save_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "model_state": model.state_dict(),
        "config":      asdict(model.config),
        "task_ids":    list(dataset.task_ids),
        "data_path":   str(data_path),
    }
    if optimizer is not None:
        checkpoint["optimizer_state"] = optimizer.state_dict()
        checkpoint["optimizer_name"]  = _optimizer_identity(optimizer)
        checkpoint["optimizer_hparams"] = _param_groups_snapshot(optimizer.param_groups)
    if scheduler  is not None: checkpoint["scheduler_state"] = scheduler.state_dict()
    if global_step is not None: checkpoint["global_step"]    = int(global_step)
    if epoch       is not None: checkpoint["epoch"]          = int(epoch)
    if rng_state   is not None: checkpoint["rng_state"]      = rng_state
    torch.save(checkpoint, save_path)
    print(f"Saved checkpoint to {save_path}")


def _normalize_checkpoint_epochs(checkpoint_epochs, total_epochs: int) -> Optional[Set[int]]:
    if checkpoint_epochs is None:
        return None
    if isinstance(checkpoint_epochs, bool):
        raise TypeError("checkpoint_epochs must be an int or list of ints, not bool.")
    if isinstance(checkpoint_epochs, int):
        interval = int(checkpoint_epochs)
        return set(range(interval, total_epochs + 1, interval)) if interval > 0 else None
    if isinstance(checkpoint_epochs, Sequence) and not isinstance(checkpoint_epochs, (str, bytes)):
        epochs: Set[int] = set()
        for item in checkpoint_epochs:
            if isinstance(item, bool):
                raise TypeError("checkpoint_epochs must be an int or list of ints, not bool.")
            try:
                epoch = int(item)
            except (TypeError, ValueError) as exc:
                raise TypeError("checkpoint_epochs must be an int or list of ints.") from exc
            if epoch <= 0:
                raise ValueError("checkpoint_epochs entries must be positive.")
            if epoch <= total_epochs:
                epochs.add(epoch)
        return epochs or None
    raise TypeError("checkpoint_epochs must be an int or list of ints.")


def _checkpoint_path_for_epoch(save_path: Path, epoch: int, total_epochs: int) -> Path:
    width  = max(2, len(str(total_epochs)))
    return save_path.with_name(f"{save_path.stem}.epoch{epoch:0{width}d}{save_path.suffix}")


# ── Main training orchestrator ────────────────────────────────────────────────

def train_model(
    args:        argparse.Namespace,
    model:       ARCTransformer,
    dataloader:  torch.utils.data.DataLoader,
    dataset:     PuzzleDataset,
    device:      torch.device,
    data_path:   Path,
    checkpoint:  Optional[Dict[str, Any]] = None,
) -> None:
    """Run the full training loop (no evaluation)."""
    if checkpoint is None:
        checkpoint = getattr(model, "_loaded_checkpoint", None)

    # ── Validation dataloader ────────────────────────────────────────────────
    do_validate    = getattr(args, "do_validate", True)
    val_dataloader = None

    sol_path = Path(data_path).parent / "solutions.json"
    if do_validate or sol_path.exists():
        val_batch_size = getattr(args, "val_batch_size", args.batch_size)
        val_dataset    = PuzzleDataset(
            json_path=data_path, splits=("test",), include_outputs=True,
            load_test_solutions=True, max_seq_len=MAX_LEN, drop_long_sequences=True,
            task_whitelist=dataset.task_ids,
        )
        val_dataloader = make_dataloader(dataset=val_dataset, batch_size=val_batch_size, shuffle=False)
        tag = "Validation" if do_validate else "Token-loss validation"
        print(f"{tag} dataset: {len(val_dataset)} examples across {len(set(e.task_id for e in val_dataset.examples))} tasks.")
    else:
        print("solutions.json not found — skipping token-loss validation.")

    # ── Logging ───────────────────────────────────────────────────────────────
    log_file       = getattr(args, "train_log_file", None)
    log_file       = Path(log_file) if log_file is not None and not isinstance(log_file, Path) else log_file
    train_log_mode = str(getattr(args, "train_log_mode", "10_steps"))
    log_location   = str(getattr(args, "log_location", "both"))
    log_handle     = None
    if log_location in ("file", "both") and log_file is not None:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        log_handle = log_file.open("a")

    # ── Save paths ────────────────────────────────────────────────────────────
    save_path = getattr(args, "save_path", None)
    save_path = Path(save_path) if save_path is not None and not isinstance(save_path, Path) else save_path
    checkpoint_schedule = _normalize_checkpoint_epochs(getattr(args, "checkpoint_epochs", None), args.epochs)

    # ── Optimizer ─────────────────────────────────────────────────────────────
    attn_wd    = getattr(args, "attention_weight_decay", args.weight_decay)
    tok_emb_wd = getattr(args, "token_embedding_weight_decay", 0.0)
    task_emb_wd = getattr(args, "task_embedding_weight_decay", 0.0)

    optimizer = _build_optimizer(args, model, device, attn_wd, tok_emb_wd, task_emb_wd)
    desired_hparams = _param_groups_snapshot(optimizer.param_groups)

    batch_sampler = getattr(dataloader, "batch_sampler", None)
    if batch_sampler is not None and hasattr(batch_sampler, "drop_last"):
        batch_sampler.drop_last = True

    step            = int(checkpoint.get("global_step", 0)) if checkpoint else 0
    steps_per_epoch = len(dataloader)

    # ── Determine start epoch ─────────────────────────────────────────────────
    start_epoch = 0
    if checkpoint:
        saved_epoch = checkpoint.get("epoch", checkpoint.get("epochs_completed"))
        if saved_epoch is not None:
            start_epoch = int(saved_epoch)
        elif step > 0 and steps_per_epoch > 0:
            start_epoch = step // steps_per_epoch
        if start_epoch > args.epochs:
            print(f"Checkpoint has {start_epoch} epochs completed; configured epochs={args.epochs}. Nothing to train.")
            start_epoch = args.epochs

    if checkpoint and step > 0:        print(f"Resuming training from global_step={step}.")
    if checkpoint and start_epoch > 0: print(f"Resuming training from epoch {start_epoch + 1}/{args.epochs}.")

    # ── LR schedule ───────────────────────────────────────────────────────────
    resume_warmup = False
    if checkpoint:
        if _optimizer_switch_detected(optimizer, checkpoint):
            resume_warmup = True
            print("Detected optimizer change; resetting LR schedule.")
        if _optimizer_hparams_changed(optimizer, checkpoint):
            resume_warmup = True
            print("Detected optimizer hyperparameter change; applying new settings.")

    if checkpoint and "optimizer_state" in checkpoint:
        restored = _load_optimizer_state(optimizer, checkpoint["optimizer_state"], device)
        if restored:
            if _optimizer_hparams_changed(optimizer, checkpoint):
                _apply_param_group_hparams(optimizer.param_groups, desired_hparams)
            print("Restored optimizer state from checkpoint.")
        else:
            resume_warmup = True
            print("Skipping optimizer state restore (incompatible optimizer).")
    elif checkpoint:
        resume_warmup = True
        print("No optimizer state in checkpoint; starting with fresh optimizer.")

    total_epochs     = max(0.0, float(args.epochs))
    steps_per_epoch  = max(1, steps_per_epoch)
    warmup_pct       = max(0.0, min(1.0, float(getattr(args, "warmup_pct", 0.02))))
    warmup_epochs    = max(0.0, min(total_epochs, total_epochs * warmup_pct))
    min_lr_factor    = max(0.0, min(1.0, float(getattr(args, "lr_floor", 0.01))))
    decay_start_pct  = max(0.0, min(1.0, float(getattr(args, "wsd_decay_start_pct", 0.8))))
    decay_start_ep   = max(min(total_epochs * decay_start_pct, total_epochs), warmup_epochs)
    decay_epochs     = max(1e-8, total_epochs - decay_start_ep)

    def base_lr_lambda(cur_epoch: float) -> float:
        if warmup_epochs > 0 and cur_epoch < warmup_epochs:
            return float(cur_epoch) / float(warmup_epochs)
        if cur_epoch < decay_start_ep:
            return 1.0
        progress = max(0.0, min(1.0, (float(cur_epoch) - decay_start_ep) / float(decay_epochs)))
        return min_lr_factor + (1.0 - min_lr_factor) * (1.0 - progress)

    lr_lambda = base_lr_lambda
    resume_start = float(start_epoch)
    if resume_warmup and resume_start > 0:
        rem = max(0.0, total_epochs - resume_start)
        if rem > 0:
            rw_epochs = max(1.0 / float(steps_per_epoch), min(rem * 0.02, rem))
            rw_end    = resume_start + rw_epochs
            rw_target = base_lr_lambda(rw_end)

            def lr_lambda(cur_epoch: float) -> float:
                if cur_epoch < resume_start:
                    return base_lr_lambda(cur_epoch)
                if cur_epoch < rw_end:
                    return rw_target * float(cur_epoch - resume_start) / float(rw_epochs)
                return base_lr_lambda(cur_epoch)

            print(f"Applying resume LR warmup for {rw_epochs:.3f} epochs.")

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    scheduler_restored = False
    if checkpoint and "scheduler_state" in checkpoint and not resume_warmup:
        sd         = checkpoint["scheduler_state"]
        last_epoch = sd.get("last_epoch") if isinstance(sd, dict) else None
        if isinstance(last_epoch, (int, float)) and abs(float(last_epoch) - float(start_epoch)) > 1.0:
            print("Skipping scheduler state restore; using epoch-based schedule from args.")
        else:
            scheduler.load_state_dict(sd)
            scheduler_restored = True
            print("Restored scheduler state from checkpoint.")
    if not scheduler_restored and start_epoch > 0:
        if resume_warmup:
            base_factor = base_lr_lambda(resume_start)
            for base_lr, group in zip(scheduler.base_lrs, optimizer.param_groups):
                group["lr"] = base_lr * base_factor
        else:
            scheduler.last_epoch = int(float(start_epoch)) - 1
            scheduler.step()

    # ── Optional torch.compile ────────────────────────────────────────────────
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
    should_compile = hasattr(torch, "compile") and device.type == "cuda" and "T4" not in gpu_name
    if should_compile:
        print("Compiling model for training speedup...")
        training_model = torch.compile(model)
    else:
        print(f"Skipping torch.compile (GPU: {gpu_name or 'cpu'}).")
        training_model = model

    augmentor   = getattr(dataloader, "augmentor", None)
    best_val    = float("inf")

    try:
        for epoch in range(start_epoch, args.epochs):
            if augmentor is not None:
                augmentor.set_epoch(epoch)
                if epoch == start_epoch or (epoch + 1) % 50 == 0:
                    print(f"  [Epoch {epoch + 1}] Augmentor epoch seed = {augmentor._epoch}, "
                          f"{len(augmentor.augments_by_key)} augmentable sequences")

            t0   = perf_counter()
            step = train_epoch(
                model=training_model, dataloader=dataloader, optimizer=optimizer,
                device=device, grad_clip=args.grad_clip,
                gradient_accumulation_steps=getattr(args, "gradient_accumulation_steps", 1),
                start_step=step, scheduler=scheduler, epoch=epoch,
                steps_per_epoch=steps_per_epoch, train_log_mode=train_log_mode,
                log_location=log_location, log_handle=log_handle,
            )
            epoch_time = perf_counter() - t0
            train_loss = val_epoch(model=model, dataloader=dataloader, device=device)
            current_lr = scheduler.get_last_lr()[0] if scheduler else optimizer.param_groups[0]["lr"]

            if val_dataloader is not None:
                vl = val_epoch(model=model, dataloader=val_dataloader, device=device)
                best_val = min(best_val, vl)
                msg = (
                    f"Epoch {epoch + 1}/{args.epochs} | lr={current_lr:.2e} | "
                    f"train_loss={train_loss:.4f} | val_loss={vl:.4f} | "
                    f"best_val={best_val:.4f} | time={epoch_time:.1f}s"
                )
            else:
                msg = (
                    f"Epoch {epoch + 1}/{args.epochs} | lr={current_lr:.2e} | "
                    f"train_loss={train_loss:.4f} | time={epoch_time:.1f}s"
                )

            print(msg)
            _emit_log(msg, log_location, log_handle)

            if checkpoint_schedule and save_path is not None:
                epoch_num = epoch + 1
                if epoch_num in checkpoint_schedule:
                    save_checkpoint(
                        model, dataset, data_path,
                        _checkpoint_path_for_epoch(save_path, epoch_num, args.epochs),
                        optimizer=optimizer, global_step=step, epoch=epoch_num,
                        rng_state=save_rng(device), scheduler=scheduler,
                    )

        save_checkpoint(
            model, dataset, data_path, save_path,
            optimizer=optimizer, global_step=step, epoch=args.epochs,
            rng_state=save_rng(device), scheduler=scheduler,
        )
    finally:
        if log_handle is not None:
            log_handle.close()


## Model & Dataset Builder


In [6]:
def load_checkpoint(checkpoint_path: Optional[Path]) -> Optional[Dict[str, Any]]:
    """Load a model checkpoint from disk."""
    if checkpoint_path is None:
        return None
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint {checkpoint_path} does not exist.")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if "model_state" not in checkpoint:
        checkpoint = {"model_state": checkpoint}
    checkpoint["__path__"] = str(checkpoint_path)
    print(f"Loaded checkpoint from {checkpoint_path}")
    return checkpoint

def infer_num_examples_from_checkpoint(
    checkpoint: Optional[Dict[str, Any]],
) -> Optional[int]:
    """Extract the number of examples from checkpoint metadata."""
    if not checkpoint:
        return None
    config = checkpoint.get("config")
    if config and "num_examples" in config:
        return int(config["num_examples"])
    state_dict = checkpoint.get("model_state", {})
    weight = state_dict.get("example_embedding.weight")
    if weight is not None:
        return int(weight.shape[0])
    return None

def setup_model_and_data(
    args: argparse.Namespace,
    checkpoint: Optional[Dict[str, Any]] = None,
    reuse_dataset: Optional[PuzzleDataset] = None,
    is_eval: bool = False,
) -> Tuple[
    ARCTransformer, PuzzleDataset, torch.utils.data.DataLoader, torch.device, Path
]:
    """Construct dataset, dataloader, and model for a given arg namespace.

    Shared by CLI entrypoints and notebooks so that training, evaluation,
    and inference can be orchestrated independently.
    """
    fix_seed(args.seed)
    device = get_device(getattr(args, "device", "cuda"))
    checkpoint = checkpoint if checkpoint is not None else load_checkpoint(args.checkpoint_path)

    data_path = args.data_path
    if data_path is None:
        if checkpoint and "data_path" in checkpoint:
            data_path = Path(checkpoint["data_path"])
        else:
            raise ValueError(
                "--data-path is required when loading saved_models that do not encode "
                "their source dataset. Please re-run with the same dataset used for training."
            )
    data_path = Path(data_path)

    checkpoint_num_examples = infer_num_examples_from_checkpoint(checkpoint)

    task_whitelist = None
    if checkpoint and "task_ids" in checkpoint:
        task_whitelist = checkpoint["task_ids"]

    if reuse_dataset is not None:
        print("Reusing existing dataset from RAM (skipping 3D pre-computation).")
        dataset = reuse_dataset
    else:
        splits = ("train", "test") if is_eval else ("train",)
        dataset = PuzzleDataset(
            json_path=data_path,
            splits=splits,
            include_outputs=True,
            max_seq_len=MAX_LEN,
            task_whitelist=task_whitelist,
            drop_long_sequences=True,
        )

    use_aug = bool(getattr(args, "enable_aug", False) and not is_eval)
    augmentor = None

    if use_aug:
        max_augments = int(getattr(args, "max_augments", 0) or 0)

        augmentor = setup_augmentor(
            dataset.examples,
            dataset.task_input_colors,
            max_augments=max_augments,
            enable_color=bool(getattr(args, "enable_color_aug", False)),
            enable_dihedral=bool(getattr(args, "enable_dihedral_aug", False)),
            seed=args.seed,
            color_apply_to_test_split=bool(getattr(args, "color_apply_to_test", False)),
            dihedral_apply_to_test_split=bool(getattr(args, "dihedral_apply_to_test", False)),
        )
        dataset.augmentor = augmentor

    collate_augment_selector = None
    if augmentor is not None:
        collate_augment_selector = augmentor.select_for_example
    use_length_bucketing = bool(is_eval or getattr(args, "eval_only", False))
    dataloader = make_dataloader(
        dataset=dataset,
        batch_size=args.batch_size,
        shuffle=not getattr(args, "eval_only", False),
        augment_selector=collate_augment_selector,
        use_length_bucketing=use_length_bucketing,
    )
    if augmentor is not None:
        dataloader.augmentor = augmentor

    if (
        checkpoint_num_examples is not None
        and dataset.num_examples != checkpoint_num_examples
    ):
        raise ValueError(
            "Dataset task-count mismatch: "
            f"checkpoint was trained with {checkpoint_num_examples} unique examples but the provided dataset "
            f"currently exposes {dataset.num_examples}. Pass the original --data-path or retrain."
        )

    if checkpoint and "config" in checkpoint:
        config = ARCTransformerConfig(**checkpoint["config"])
    else:
        num_examples = checkpoint_num_examples or max(1, dataset.num_examples)
        d_model = getattr(args, "d_model", 128)
        n_heads = getattr(args, "n_heads", 4)
        d_ff = getattr(args, "d_ff", 2560)
        n_layers = getattr(args, "n_layers", 4)
        dropout = getattr(args, "dropout", 0.1)
        attention_dropout = getattr(args, "attention_dropout", None)
        config = ARCTransformerConfig(
            num_examples=num_examples,
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            dropout=dropout,
            attention_dropout=attention_dropout,
        )

    if dataset.num_examples != config.num_examples:
        raise ValueError(
            f"Dataset provides {dataset.num_examples} examples but model expects "
            f"{config.num_examples}. Please ensure the dataset/task whitelist matches the checkpoint."
        )

    model = ARCTransformer(config).to(device)

    if checkpoint:
        state_dict = checkpoint["model_state"]
        incompatible = model.load_state_dict(state_dict, strict=False)
        if "dihedral_embedding.weight" in set(incompatible.missing_keys):
            checkpoint_path = checkpoint.get("__path__", "<checkpoint>")
            raise ValueError(
                "Checkpoint is missing required dihedral embedding weights "
                f"(dihedral_embedding.weight): {checkpoint_path}. "
                "This checkpoint predates dihedral embedding support."
            )
        load_rng(checkpoint.get("rng_state"), device)

    model._loaded_checkpoint = checkpoint

    return model, dataset, dataloader, device, data_path


## Inference & Evaluation Pipeline


In [7]:
MAX_NEW_TOKS = 931

@torch.compile(mode="default", fullgraph=True)
def _compiled_grid_update(state, token_ids, start_id, sep_id, end_id, nl_id):
    x, y, z = state.unbind(-1)
    pos_x = torch.clamp(x, min=0, max=30)
    pos_y = torch.clamp(y, min=0, max=29)
    pos_z = z

    is_start = token_ids == start_id
    is_sep = token_ids == sep_id
    is_end = token_ids == end_id
    is_newline = token_ids == nl_id

    zeros = torch.zeros_like(x)
    pos_x = torch.where(is_start | is_sep | is_end, zeros, pos_x)
    pos_y = torch.where(is_start | is_sep | is_end, zeros, pos_y)
    pos_z = torch.where(is_start, zeros, pos_z)
    pos_z = torch.where(is_sep, torch.full_like(pos_z, 2), pos_z)
    pos_z = torch.where(is_end, torch.full_like(pos_z, 4), pos_z)

    next_x = x + 1
    next_y = y
    next_z = z
    next_x = torch.where(is_newline, zeros, next_x)
    next_y = torch.where(is_newline, y + 1, next_y)
    next_x = torch.where(is_sep, zeros, next_x)
    next_y = torch.where(is_sep, zeros, next_y)
    next_z = torch.where(is_sep, torch.full_like(next_z, 3), next_z)
    next_x = torch.where(is_end | is_start, x, next_x)
    next_y = torch.where(is_end | is_start, y, next_y)
    next_z = torch.where(is_start, z, next_z)
    next_z = torch.where(is_end, z, next_z)

    return torch.stack([next_x, next_y, next_z], dim=-1), torch.stack([pos_x, pos_y, pos_z], dim=-1)

class GridTracker:
    def __init__(self, initial_state: torch.Tensor) -> None:
        self.state = initial_state.long()

    def update(self, token_ids: torch.Tensor) -> torch.Tensor:
        token_ids = token_ids.view(-1).to(device=self.state.device)
        self.state, positions = _compiled_grid_update(
            self.state, token_ids, START_ID, SEP_ID, END_ID, NL_ID
        )
        return positions

def _select_next_token(logits: torch.Tensor, temperature: Optional[float] = None, top_k: Optional[int] = None) -> torch.Tensor:
    last_logits = logits[:, -1, :]
    if (temperature is None and top_k is None) or (temperature is not None and temperature <= 0):
        return torch.argmax(last_logits, dim=-1)
    if temperature is None:
        temperature = 1.0
    use_top_k = top_k is not None and top_k > 0
    if use_top_k:
        top_k = int(top_k)
        if top_k == 1:
            return torch.argmax(last_logits, dim=-1)
        top_k = min(top_k, last_logits.size(-1))
        top_values, top_indices = torch.topk(last_logits, top_k, dim=-1)
        scaled = (top_values / temperature).float()
        probs = torch.softmax(scaled, dim=-1)
        next_index = torch.multinomial(probs, num_samples=1)
        return top_indices.gather(-1, next_index).squeeze(-1)
    scaled = (last_logits / temperature).float()
    probs = torch.softmax(scaled, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

def _left_pad_sequences(sequences: Sequence[Sequence[int]], pad_token_id: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    batch_size = len(sequences)
    max_len = max(len(seq) for seq in sequences)
    input_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long, device=device)
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.bool, device=device)
    for idx, seq in enumerate(sequences):
        seq_len = len(seq)
        start = max_len - seq_len
        input_ids[idx, start:] = torch.tensor(seq, dtype=torch.long, device=device)
        attention_mask[idx, start:] = True
    return input_ids, attention_mask

def _pad_cached_positions(cached_positions: Sequence[torch.Tensor], max_len: int, device: torch.device) -> torch.Tensor:
    positions = torch.zeros((len(cached_positions), max_len, 3), dtype=torch.long, device=device)
    for idx, pos in enumerate(cached_positions):
        seq_len = pos.size(0)
        start = max_len - seq_len
        positions[idx, start:] = pos.to(device=device, dtype=torch.long)
    return positions

def _derive_initial_state_from_prompt(input_ids: torch.Tensor, positions_3d: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    last_tokens = input_ids[:, -1]
    last_positions = positions_3d[:, -1]
    x, y, z = last_positions.unbind(-1)

    next_x = x + 1
    next_y = y
    next_z = z
    next_x = torch.where(last_tokens == NL_ID, torch.zeros_like(next_x), next_x)
    next_y = torch.where(last_tokens == NL_ID, y + 1, next_y)
    next_x = torch.where(last_tokens == SEP_ID, torch.zeros_like(next_x), next_x)
    next_y = torch.where(last_tokens == SEP_ID, torch.zeros_like(next_y), next_y)
    next_z = torch.where(last_tokens == SEP_ID, torch.full_like(next_z, 3), next_z)
    next_x = torch.where(last_tokens == END_ID, x, next_x)
    next_y = torch.where(last_tokens == END_ID, y, next_y)
    next_z = torch.where(last_tokens == END_ID, z, next_z)
    next_x = torch.where(last_tokens == START_ID, torch.zeros_like(next_x), next_x)
    next_y = torch.where(last_tokens == START_ID, torch.zeros_like(next_y), next_y)
    next_z = torch.where(last_tokens == START_ID, torch.ones_like(next_z), next_z)

    initial_state = torch.stack([next_x, next_y, next_z], dim=-1)
    finished = last_tokens == END_ID
    return initial_state, finished

@torch.inference_mode()
def batch_generate(
    model, prompts, example_ids, dihedral_ids, device, max_new_tokens=931,
    cached_positions=None, temperature: Optional[float] = None, top_k: Optional[int] = None,
):
    model.eval()
    batch_size = len(prompts)

    example_ids_tensor = torch.tensor(example_ids, dtype=torch.long, device=device)
    dihedral_ids_tensor = torch.tensor(dihedral_ids, dtype=torch.long, device=device)
    input_ids, attention_mask = _left_pad_sequences(prompts, END_ID, device)
    current_len = input_ids.size(1)
    batch_max_needed = current_len + max_new_tokens
    batch_max_needed = (batch_max_needed + 127) // 128 * 128
    max_model_len = min(batch_max_needed, model.config.max_seq_len)

    if cached_positions and all(p is not None for p in cached_positions):
        prompt_positions = _pad_cached_positions([p for p in cached_positions if p is not None], input_ids.size(1), device)
    else:
        prompt_positions = get_3d_positions(input_ids, attention_mask).to(device=device, dtype=torch.long)

    initial_state, finished = _derive_initial_state_from_prompt(input_ids, prompt_positions)
    grid_state = GridTracker(initial_state)
    example_embeds = model.example_embedding(example_ids_tensor).to(dtype=torch.bfloat16)
    current_len = input_ids.size(1)

    full_attention_mask = torch.zeros((batch_size, max_model_len), dtype=torch.bool, device=device)
    full_attention_mask[:, :current_len] = attention_mask

    outputs = model.forward_generate(
        input_ids=input_ids, example_ids=example_ids_tensor, dihedral_ids=dihedral_ids_tensor, past_key_values=None,
        positions_3d=prompt_positions, attention_mask=attention_mask, example_embeds=example_embeds,
    )
    logits = outputs["logits"]
    prompt_kvs = outputs["past_key_values"]

    past_key_values = []
    for k, v in prompt_kvs:
        B, H, L, D = k.shape
        k_buf = torch.zeros((B, H, max_model_len, D), dtype=torch.bfloat16, device=device)
        v_buf = torch.zeros((B, H, max_model_len, D), dtype=torch.bfloat16, device=device)
        k_buf[:, :, :L, :] = k
        v_buf[:, :, :L, :] = v
        past_key_values.append((k_buf, v_buf))
    past_key_values = tuple(past_key_values)

    if not hasattr(model, "_compiled_decode"):
        print("Bypassing model compile for decoding step...")
        model._compiled_decode = model.forward_generate

    cache_position = torch.tensor([current_len], dtype=torch.long, device=device)
    steps_remaining = min(max_new_tokens, max_model_len - current_len)
    generated_tokens_buffer = torch.full((batch_size, steps_remaining), END_ID, dtype=torch.long, device=device)

    for step_i in range(steps_remaining):
        if finished.all():
            break
        torch.compiler.cudagraph_mark_step_begin()
        next_token = _select_next_token(logits, temperature=temperature, top_k=top_k)
        next_token = torch.where(finished, torch.tensor(END_ID, device=device), next_token)
        generated_tokens_buffer[:, step_i] = next_token
        finished = finished | (next_token == END_ID)
        token_positions = grid_state.update(next_token).unsqueeze(1)
        active_rows = ~finished
        full_attention_mask[:, cache_position] = active_rows.unsqueeze(1)
        decode_block_mask = None
        if hasattr(model, "build_decode_block_mask_for_step"):
            decode_block_mask = model.build_decode_block_mask_for_step(
                attention_mask=full_attention_mask,
                cache_position=cache_position,
                kv_len=max_model_len,
                device=device,
            )
        outputs = model._compiled_decode(
            input_ids=next_token.unsqueeze(1), example_ids=example_ids_tensor, dihedral_ids=dihedral_ids_tensor,
            past_key_values=past_key_values, positions_3d=token_positions,
            attention_mask=full_attention_mask, cache_position=cache_position, example_embeds=example_embeds,
            decode_block_mask=decode_block_mask,
        )
        logits = outputs["logits"]
        cache_position.add_(1)

    generated_cpu = generated_tokens_buffer.tolist()
    results = []
    for i, prompt in enumerate(prompts):
        gen_seq = []
        for token in generated_cpu[i]:
            gen_seq.append(token)
            if token == END_ID:
                break
        results.append(list(prompt) + gen_seq)
    return results

def _build_prompt_from_tokens(tokens: Sequence[int]) -> List[int]:
    if SEP_ID not in tokens:
        raise ValueError("Prompt sequence is missing <input_output_separator>.")
    sep_idx = tokens.index(SEP_ID)
    return list(tokens[: sep_idx + 1])

def _select_tokens_for_example(example: object, transform_index: Optional[int]) -> Tuple[List[int], Optional[torch.Tensor]]:
    tokens = getattr(example, "tokens")
    cached_positions = getattr(example, "cached_positions", None)
    if transform_index is not None:
        tokens_by_dihedral = getattr(example, "tokens_by_dihedral", None)
        if tokens_by_dihedral:
            if transform_index < 0 or transform_index >= len(tokens_by_dihedral):
                raise ValueError(f"Invalid dihedral transform index {transform_index}.")
            tokens = tokens_by_dihedral[transform_index]
            cached_by_dihedral = getattr(example, "cached_positions_by_dihedral", None)
            if cached_by_dihedral:
                cached_positions = cached_by_dihedral[transform_index]
    token_list = tokens.tolist() if isinstance(tokens, torch.Tensor) else list(tokens)
    return token_list, cached_positions

def _sequence_length_for_example(example: object, transform_index: Optional[int]) -> int:
    if transform_index is not None:
        tokens_by_dihedral = getattr(example, "tokens_by_dihedral", None)
        if tokens_by_dihedral:
            return int(tokens_by_dihedral[transform_index].size(0))
    seq_len = getattr(example, "seq_len", None)
    if seq_len is not None:
        return int(seq_len)
    tokens = getattr(example, "tokens")
    return int(tokens.size(0) if isinstance(tokens, torch.Tensor) else len(tokens))

def _prepare_examples_for_inference(
    examples: Sequence[object],
    color_mappings: Optional[Sequence[Optional[Sequence[int]]]] = None,
    color_apply_fn: Optional[Callable[[str], bool]] = None,
    dihedral_transform_indices: Optional[Sequence[Optional[int]]] = None,
    pair_indices: Optional[Sequence[int]] = None,
) -> Tuple[
    List[List[int]],
    List[int],
    List[int],
    List[Dict[str, object]],
    List[Optional[torch.Tensor]],
]:
    prompts: List[List[int]] = []
    example_ids: List[int] = []
    dihedral_ids: List[int] = []
    metadata: List[Dict[str, object]] = []
    cached_positions: List[Optional[torch.Tensor]] = []

    for idx, ex in enumerate(examples):
        if not hasattr(ex, "tokens"):
            raise ValueError("Examples must provide a 'tokens' attribute.")
        transform_index = dihedral_transform_indices[idx] if dihedral_transform_indices is not None else None
        dihedral_id = int(transform_index) if transform_index is not None else 0
        if dihedral_id < 0 or dihedral_id >= 8:
            raise ValueError(f"Invalid dihedral transform index {dihedral_id}.")
        raw_tokens, cached = _select_tokens_for_example(ex, transform_index)
        split = getattr(ex, "split", None)
        mapping = color_mappings[idx] if color_mappings is not None else None
        should_color = mapping is not None and (color_apply_fn is None or color_apply_fn(split))
        tokens = permute_token_colors(raw_tokens, mapping) if should_color else raw_tokens
        prompt_tokens = _build_prompt_from_tokens(tokens)
        prompts.append(prompt_tokens)
        example_ids.append(int(getattr(ex, "example_id", 0)))
        dihedral_ids.append(dihedral_id)
        if cached is not None:
            cached_positions.append(cached[: len(prompt_tokens)])
        else:
            cached_positions.append(None)

        pair_index = pair_indices[idx] if pair_indices is not None else getattr(ex, "pair_index", None)
        metadata.append({
            "task_id": getattr(ex, "task_id", None),
            "pair_index": pair_index,
            "example_id": getattr(ex, "example_id", None),
            "split": getattr(ex, "split", None),
        })

    return prompts, example_ids, dihedral_ids, metadata, cached_positions

def _build_generation_results(
    sequences: Sequence[Sequence[int]],
    metadata: Sequence[Dict[str, object]],
    prompts: Sequence[Sequence[int]],
) -> List[Dict[str, object]]:
    results: List[Dict[str, object]] = []
    for seq, meta, prompt in zip(sequences, metadata, prompts):
        output_tokens = get_output_tokens(seq)
        predicted_grid = tok_to_grid(output_tokens)
        result = {
            "task_id": meta.get("task_id"),
            "pair_index": meta.get("pair_index"),
            "example_id": meta.get("example_id"),
            "split": meta.get("split"),
            "prompt_tokens": list(prompt),
            "sequence": list(seq),
            "output_tokens": output_tokens,
            "output_grid": predicted_grid,
        }
        results.append(result)
    return results

def _run_generation_batch(
    model: ARCTransformer,
    prompts: Sequence[Sequence[int]],
    example_ids: Sequence[int],
    dihedral_ids: Sequence[int],
    metadata: Sequence[Dict[str, object]],
    cached_positions: Sequence[Optional[torch.Tensor]],
    device: torch.device,
    max_new_tokens: int,
    temperature: Optional[float] = None,
    top_k: Optional[int] = None,
) -> List[Dict[str, object]]:
    sequences = batch_generate(
        model=model, prompts=prompts, example_ids=example_ids, dihedral_ids=dihedral_ids, device=device,
        max_new_tokens=max_new_tokens, cached_positions=cached_positions,
        temperature=temperature, top_k=top_k,
    )
    return _build_generation_results(sequences=sequences, metadata=metadata, prompts=prompts)

def _gather_examples_for_split(
    dataset, split: str, task_ids: Optional[Sequence[str]] = None, pair_index: Optional[int] = None,
):
    examples = []
    for example in dataset.iter_examples(split=split):
        if task_ids is not None and example.task_id not in task_ids:
            continue
        if pair_index is not None and example.pair_index != pair_index:
            continue
        examples.append(example)
    return examples

def _identity_mapping() -> List[int]:
    return list(range(N_VOCAB))

def _split_allows_color(augmentor: Augmentor, split: str) -> bool:
    if split == "test":
        return bool(augmentor.color_apply_to_test_split)
    return True

def _split_allows_dihedral(augmentor: Augmentor, split: str) -> bool:
    if split == "test":
        return bool(augmentor.dihedral_apply_to_test_split)
    return True

def _allowed_tuple_indices(augments: Augments, *, allow_color: bool, allow_dihedral: bool) -> List[int]:
    indices: List[int] = []
    for idx, (d_idx, c_idx) in enumerate(zip(augments.dihedral_indices, augments.color_map_indices)):
        if not allow_color and c_idx != 0:
            continue
        if not allow_dihedral and d_idx != 0:
            continue
        indices.append(idx)
    if not indices:
        indices = [augments.identity_tuple_index]
    return indices

def _build_color_mappings_by_task(
    examples: Sequence[object], augmentor: Augmentor, split: str,
) -> Tuple[Dict[str, List[List[int]]], Dict[str, Dict[Tuple[int, ...], int]]]:
    mappings_by_task: Dict[str, List[List[int]]] = {}
    mapping_index_by_task: Dict[str, Dict[Tuple[int, ...], int]] = {}
    identity = _identity_mapping()
    identity_key = tuple(identity)
    allow_color = _split_allows_color(augmentor, split)
    allow_dihedral = _split_allows_dihedral(augmentor, split)

    for ex in examples:
        task_id = getattr(ex, "task_id", None)
        if task_id is None:
            continue
        if task_id not in mappings_by_task:
            mappings_by_task[task_id] = [identity]
            mapping_index_by_task[task_id] = {identity_key: 0}

    for ex in examples:
        task_id = getattr(ex, "task_id", None)
        if task_id is None:
            continue
        key = (task_id, getattr(ex, "split", None), getattr(ex, "pair_index", None))
        augments = augmentor.augments_by_key.get(key)
        if augments is None:
            continue
        allowed = _allowed_tuple_indices(augments, allow_color=allow_color, allow_dihedral=allow_dihedral)
        for idx in allowed:
            map_idx = augments.color_map_indices[idx]
            mapping = augments.color_maps[map_idx].tolist()
            mapping_key = tuple(mapping)
            lookup = mapping_index_by_task[task_id]
            if mapping_key not in lookup:
                lookup[mapping_key] = len(mappings_by_task[task_id])
                mappings_by_task[task_id].append(mapping)
    return mappings_by_task, mapping_index_by_task

@torch.inference_mode()
def infer_split(
    model: ARCTransformer, dataset, split: str, device: torch.device,
    batch_size: int = 16, max_new_tokens: int = MAX_NEW_TOKS,
    task_ids: Optional[Sequence[str]] = None, pair_index: Optional[int] = None,
    temperature: Optional[float] = None, top_k: Optional[int] = None,
    augmentor: Optional[Augmentor] = None,
) -> Tuple[List[Dict[str, object]], Dict[str, List[List[int]]], bool]:
    model.eval()
    if next(model.parameters()).dtype != torch.bfloat16:
        model.to(dtype=torch.bfloat16)

    examples = _gather_examples_for_split(
        dataset, split=split, task_ids=task_ids, pair_index=pair_index,
    )
    if not examples:
        return [], {}, False

    work_items = []
    color_mappings_by_task: Dict[str, List[List[int]]] = {}
    dihedral_augmented = False
    pack_dihedral = False

    if augmentor is None:
        for ex in examples:
            work_items.append((ex, None, None, 0, None, ex.pair_index, _sequence_length_for_example(ex, None)))
    else:
        allow_color = _split_allows_color(augmentor, split)
        allow_dihedral = _split_allows_dihedral(augmentor, split)
        color_mappings_by_task, mapping_index_by_task = _build_color_mappings_by_task(examples, augmentor, split)
        for ex in examples:
            task_id = getattr(ex, "task_id", None)
            if task_id is None:
                continue
            key = (task_id, getattr(ex, "split", None), getattr(ex, "pair_index", None))
            augments = augmentor.augments_by_key.get(key)
            if augments is None:
                continue
            allowed = _allowed_tuple_indices(augments, allow_color=allow_color, allow_dihedral=allow_dihedral)
            for idx in allowed:
                dihedral_idx = int(augments.dihedral_indices[idx])
                color_map_idx = int(augments.color_map_indices[idx])
                mapping = augments.color_maps[color_map_idx].tolist()
                mapping_key = tuple(mapping)
                mapping_idx = mapping_index_by_task[task_id].get(mapping_key, 0)
                seq_len = _sequence_length_for_example(ex, dihedral_idx)
                base_pair_index = getattr(ex, "pair_index", None)
                work_items.append((ex, dihedral_idx, dihedral_idx, mapping_idx, mapping, base_pair_index, seq_len))
                if dihedral_idx != 0:
                    dihedral_augmented = True
        pack_dihedral = allow_dihedral and dihedral_augmented

    work_items.sort(key=lambda item: item[-1], reverse=True)
    all_results: List[Dict[str, object]] = []

    for start in range(0, len(work_items), batch_size):
        chunk = work_items[start : start + batch_size]
        batch_examples = [item[0] for item in chunk]
        batch_transform_indices = [item[2] for item in chunk]
        batch_c_indices = [item[3] for item in chunk]
        batch_mappings = [item[4] for item in chunk]
        batch_pair_indices = []
        for item in chunk:
            base_pair_index = item[5]
            if augmentor is not None and pack_dihedral and base_pair_index is not None:
                batch_pair_indices.append(int(base_pair_index) * 8 + item[1])
            else:
                batch_pair_indices.append(base_pair_index)

        if augmentor is None:
            prompts, example_ids, dihedral_ids, metadata, cached_positions = _prepare_examples_for_inference(batch_examples)
        else:
            prompts, example_ids, dihedral_ids, metadata, cached_positions = _prepare_examples_for_inference(
                batch_examples,
                color_mappings=batch_mappings, color_apply_fn=None,
                dihedral_transform_indices=batch_transform_indices, pair_indices=batch_pair_indices,
            )

        batch_results = _run_generation_batch(
            model=model, prompts=prompts, example_ids=example_ids, dihedral_ids=dihedral_ids, metadata=metadata,
            cached_positions=cached_positions, device=device, max_new_tokens=max_new_tokens,
            temperature=temperature, top_k=top_k,
        )

        print(f"[{split}] Finished batch {start // batch_size + 1} / {(len(work_items) + batch_size - 1) // batch_size}")

        if augmentor is None:
            all_results.extend(batch_results)
        else:
            for res, c_idx, d_idx in zip(batch_results, batch_c_indices, batch_transform_indices):
                res["color_permutation_index"] = c_idx
                res["dihedral_index"] = d_idx
                all_results.append(res)

    return all_results, color_mappings_by_task, dihedral_augmented

def _grid_to_tuple(grid: Sequence[Sequence[int]]) -> Tuple[Tuple[int, ...], ...]:
    return tuple(tuple(int(val) for val in row) for row in grid)

def _tuple_to_grid(grid_tuple: Tuple[Tuple[int, ...], ...]) -> List[List[int]]:
    return [list(row) for row in grid_tuple]

@dataclass
class VoteResult:
    task_id: str
    original_pair_index: int
    selected_outputs: List[List[List[int]]]
    ranked_candidates: List[Dict[str, object]]
    num_generated: int
    num_valid: int
    discarded_non_rectangular: int
    discarded_input_copies: int

def vote_predictions(
    results: Sequence[Dict[str, object]],
    top_k: int = 2,
    discard_input_copies: bool = True,
    rng: Optional[random.Random] = None,
    is_dihedral_augmented: bool = False,
    color_mappings_by_task: Optional[Dict[str, Sequence[Sequence[int]]]] = None,
) -> List[VoteResult]:
    """Aggregate augmented predictions via AAIVR voting."""
    rng = rng if rng is not None else random
    case_map: Dict[Tuple[str, int], Dict[str, object]] = {}

    inverse_color_mappings_by_task: Dict[str, List[List[int]]] = {}
    if color_mappings_by_task is not None:
        for task_id, mappings in color_mappings_by_task.items():
            inv_list: List[List[int]] = []
            for mapping in mappings:
                fwd = mapping if isinstance(mapping, torch.Tensor) else torch.tensor(mapping, dtype=torch.long)
                inv = torch.zeros_like(fwd)
                inv[fwd] = torch.arange(len(fwd), dtype=torch.long, device=fwd.device)
                inv_list.append(inv.tolist())
            inverse_color_mappings_by_task[task_id] = inv_list

    for res in results:
        task_id = res.get("task_id")
        pair_index = res.get("pair_index")
        if task_id is None or pair_index is None:
            continue

        if is_dihedral_augmented:
            base_pair_index = int(pair_index) // 8
            transform_index = int(pair_index) % 8
        else:
            base_pair_index = int(pair_index)
            transform_index = 0

        color_idx = res.get("color_permutation_index", 0)
        predicted_grid = res.get("output_grid", [])
        prompt_tokens = res.get("prompt_tokens", [])
        input_grids = split_grids(prompt_tokens)
        input_grid = input_grids[0] if input_grids else []

        key = (task_id, base_pair_index)
        if key not in case_map:
            case_map[key] = {"counts": {}, "generated": 0, "valid": 0, "dropped_rect": 0, "dropped_input": 0}
        stats = case_map[key]
        stats["generated"] += 1

        if not is_valid_grid(predicted_grid):
            stats["dropped_rect"] += 1
            continue
        if discard_input_copies and input_grid and predicted_grid == input_grid:
            stats["dropped_input"] += 1
            continue

        try:
            normalized_grid = rotate_grid_inverse(predicted_grid, transform_index)
            inv_list = inverse_color_mappings_by_task.get(task_id, [])
            if inv_list and color_idx > 0 and color_idx < len(inv_list):
                normalized_grid = permute_grid_colors(normalized_grid, inv_list[color_idx])
        except Exception:
            stats["dropped_rect"] += 1
            continue

        if not is_valid_grid(normalized_grid):
            stats["dropped_rect"] += 1
            continue

        stats["valid"] += 1
        grid_key = _grid_to_tuple(normalized_grid)
        counts: Dict[Tuple[Tuple[int, ...], ...], int] = stats["counts"]
        counts[grid_key] = counts.get(grid_key, 0) + 1

    selections: List[VoteResult] = []
    for (task_id, base_idx), stats in sorted(case_map.items()):
        items = list(stats["counts"].items())
        if items:
            rng.shuffle(items)
            items.sort(key=lambda pair: pair[1], reverse=True)
        ranked_candidates = [{"grid": _tuple_to_grid(grid_key), "count": count} for grid_key, count in items]
        selected_outputs = [entry["grid"] for entry in ranked_candidates[:top_k]]

        selections.append(VoteResult(
            task_id=task_id, original_pair_index=base_idx, selected_outputs=selected_outputs,
            ranked_candidates=ranked_candidates, num_generated=stats["generated"], num_valid=stats["valid"],
            discarded_non_rectangular=stats["dropped_rect"], discarded_input_copies=stats["dropped_input"],
        ))

    return selections

def summarize_results(results: Sequence[Dict[str, object]]) -> Dict[str, object]:
    """Return basic statistics about generated results."""
    return {"total_sequences": len(results)}

class DualLogger:
    def __init__(self, filepath: Path) -> None:
        self.terminal = sys.stdout
        self.log = Path(filepath).open("w")

    def write(self, message: str) -> None:
        self.terminal.write(message)
        self.log.write(message)

    def flush(self) -> None:
        self.terminal.flush()
        self.log.flush()

    def close(self) -> None:
        self.log.close()

def _make_submission(selections: Sequence[VoteResult]) -> Dict[str, List[Dict[str, object]]]:
    submission_data: Dict[str, List[Dict[str, object]]] = {}
    temp_grouping: Dict[str, Dict[int, Dict[str, object]]] = {}

    for item in selections:
        task_id = item.task_id
        pair_idx = item.original_pair_index
        if task_id not in temp_grouping:
            temp_grouping[task_id] = {}
        top_grids = item.selected_outputs[:2]
        if not top_grids:
            top_grids = [[[0]]]
        pair_dict = {"attempt_1": top_grids[0], "attempt_2": top_grids[1] if len(top_grids) > 1 else top_grids[0]}
        temp_grouping[task_id][pair_idx] = pair_dict

    for task_id, pairs_map in temp_grouping.items():
        sorted_indices = sorted(pairs_map.keys())
        submission_data[task_id] = [pairs_map[idx] for idx in sorted_indices]

    return submission_data

def run_eval_pipeline(
    cfg: argparse.Namespace,
    *,
    run_name: str,
    max_augments: int,
    data_path: Path,
    checkpoint_path: Path,
    batch_size: int = 1300,
    splits: Sequence[str] = ("test",),
    task_ids: Optional[Sequence[str]] = None,
    timing_path: Path = Path("saved_models/timing.txt"),
) -> Tuple[str, Dict[str, Dict[str, object]], Path]:
    """Run evaluation on a single configuration, generating submission.json.

    Args:
        cfg: Training config namespace (used for augmentation settings).
        run_name: Name for this evaluation run (creates saved_models/eval_runs/<run_name>/ directory).
        max_augments: Number of augmented variants per example for test-time augmentation (TTA).
            Higher values = more diverse predictions for AAIVR voting, but slower inference.
        data_path: Path to challenges.json file.
        checkpoint_path: Path to model checkpoint (.pt file).
        batch_size: Batch size for inference.
        splits: Dataset splits to evaluate (default: ["test"]).
        task_ids: Optional list of specific task IDs to evaluate (None = all tasks).
        timing_path: Path to write timing information.

    Returns:
        Tuple of (run_name, evaluation_results, submission_path).
    """
    timing_path = Path(timing_path)
    timing_path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint_path = Path(checkpoint_path)
    data_path = Path(data_path)
    t_start = perf_counter()

    print(f"\n{'=' * 60}")
    mode_label = "Augmented" if getattr(cfg, "enable_aug", False) else "No-aug"
    print(f"STARTING EVALUATION: {run_name} ({mode_label} augs: {max_augments})")
    print(f"{'=' * 60}\n")

    base_run_dir = Path("saved_models") / "eval_runs" / run_name
    base_run_dir.mkdir(parents=True, exist_ok=True)
    eval_log_path = base_run_dir / "eval_log.txt"
    aaivr_log_path = base_run_dir / "aaivr.txt"
    submission_path = base_run_dir / "submission.json"

    # Temporarily override cfg values for this evaluation
    prev_checkpoint = getattr(cfg, "checkpoint_path", None)
    prev_data_path = getattr(cfg, "data_path", None)
    prev_aug = getattr(cfg, "max_augments", 0)

    cfg.checkpoint_path = checkpoint_path
    cfg.data_path = data_path
    use_aug = bool(getattr(cfg, "enable_aug", False))
    if use_aug:
        cfg.max_augments = int(max_augments)

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    print("Building model and dataloader...")
    model, dataset, _, device, _ = setup_model_and_data(
        cfg, checkpoint=checkpoint, is_eval=True
    )

    def log_eval(msg: str) -> None:
        print(msg)
        with eval_log_path.open("a") as handle:
            handle.write(msg + "\n")

    color_mappings_by_split: Dict[str, Dict[str, List[List[int]]]] = {}
    dihedral_by_split: Dict[str, bool] = {}

    if use_aug:
        augmentor = getattr(dataset, "augmentor", None)
        if augmentor is None or getattr(augmentor, "max_augments", None) != max_augments:
            enable_color = bool(getattr(cfg, "enable_color_aug", False))
            enable_dihedral = bool(getattr(cfg, "enable_dihedral_aug", False))
            color_apply_to_test = bool(getattr(cfg, "color_apply_to_test", False))
            dihedral_apply_to_test = bool(getattr(cfg, "dihedral_apply_to_test", False))

            augmentor = setup_augmentor(
                dataset.examples, dataset.task_input_colors, max_augments=max_augments,
                enable_color=enable_color, enable_dihedral=enable_dihedral, seed=int(cfg.seed),
                color_apply_to_test_split=color_apply_to_test, dihedral_apply_to_test_split=dihedral_apply_to_test,
            )
            dataset.augmentor = augmentor

        evaluation: Dict[str, Dict[str, object]] = {}
        for split in splits:
            split_results, color_maps, dihedral_augmented = infer_split(
                model=model, dataset=dataset, split=split, device=device, augmentor=augmentor,
                batch_size=batch_size, max_new_tokens=MAX_NEW_TOKS, task_ids=task_ids,
                temperature=getattr(cfg, "inference_temperature", None), top_k=getattr(cfg, "inference_top_k", None),
            )
            summary = summarize_results(split_results)
            evaluation[split] = {"results": split_results, "summary": summary}
            color_mappings_by_split[split] = color_maps
            dihedral_by_split[split] = dihedral_augmented
        evaluation["_aug"] = {"color_mappings_by_split": color_mappings_by_split, "dihedral_augmented_by_split": dihedral_by_split}
    else:
        evaluation: Dict[str, Dict[str, object]] = {}
        for split in splits:
            split_results, _, _ = infer_split(
                model=model, dataset=dataset, split=split, device=device, batch_size=batch_size,
                max_new_tokens=MAX_NEW_TOKS, task_ids=task_ids,
                temperature=getattr(cfg, "inference_temperature", None), top_k=getattr(cfg, "inference_top_k", None),
            )
            summary = summarize_results(split_results)
            evaluation[split] = {"results": split_results, "summary": summary}

    dihedral_augmented = dihedral_by_split.get("test", False) if use_aug else False

    for split in splits:
        summary = evaluation.get(split, {}).get("summary", {})
        total = summary.get("total_sequences", 0)
        log_eval(f"Split: {split} | Generated {total} sequences")

    print(f"Running AAIVR for {run_name}...")
    if hasattr(sys.stdout, "log"):
        sys.stdout = sys.stdout.terminal
    sys.stdout = DualLogger(aaivr_log_path)

    try:
        test_results = evaluation.get("test", {}).get("results", [])
        aaivr_results: List[VoteResult] = []
        if test_results:
            aaivr_color_mappings = color_mappings_by_split.get("test") if use_aug else None
            aaivr_results = vote_predictions(
                test_results, is_dihedral_augmented=dihedral_augmented, color_mappings_by_task=aaivr_color_mappings,
            )
            print(f"AAIVR aggregated {len(test_results)} predictions into {len(aaivr_results)} submission entries")
        else:
            print("No test results for AAIVR.")
    finally:
        if hasattr(sys.stdout, "terminal"):
            sys.stdout.close()
            sys.stdout = sys.stdout.terminal

    print(f"Generating submission.json for {run_name}...")
    submission_data = _make_submission(aaivr_results)
    with submission_path.open("w") as handle:
        json.dump(submission_data, handle)

    t_duration = perf_counter() - t_start
    print(f"Finished {run_name}. Submission saved to {submission_path}")
    print(f"Evaluation took {t_duration:.2f}s")
    with timing_path.open("a") as handle:
        handle.write(f"Evaluation {run_name}: {t_duration:.4f} s\n")

    # Restore cfg values
    cfg.checkpoint_path = prev_checkpoint
    cfg.data_path = prev_data_path
    cfg.max_augments = prev_aug

    return (run_name, evaluation, submission_path)

## Scoring & Visualisation


In [8]:
GRID_COLORS = [
    "#000000",  # 0 black
    "#0074D9",  # 1 blue
    "#FF4136",  # 2 red
    "#2ECC40",  # 3 green
    "#FFDC00",  # 4 yellow
    "#AAAAAA",  # 5 gray
    "#F012BE",  # 6 fuchsia
    "#FF851B",  # 7 orange
    "#7FDBFF",  # 8 aqua
    "#B10DC9",  # 9 purple
]

def show_grids(
    grids: List[List[List[int]]], title: Optional[str] = None, figsize=(4, 4)
) -> None:
    """Plot one or more integer grids using a fixed 10-color palette."""
    if plt is None or ListedColormap is None:
        raise RuntimeError("matplotlib is not available in this environment.")

    n = max(1, len(grids))
    fig, axes = plt.subplots(1, n, figsize=(figsize[0] * n, figsize[1]))
    if n == 1:
        axes = [axes]
    cmap = ListedColormap(GRID_COLORS)

    for ax, grid in zip(axes, grids):
        if not grid:
            ax.axis("off")
            continue
        arr = np.array(grid, dtype=int)
        ax.imshow(arr, cmap=cmap, vmin=0, vmax=9)
        ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
        ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=0.5)
        ax.tick_params(which="minor", bottom=False, left=False)
        ax.set_xticks([])
        ax.set_yticks([])
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def show_predictions(
    submission_file: Path,
    solutions_file: Optional[Path] = None,
    mode: str = "submission",
) -> None:
    """Visualize submission attempts, optionally comparing against solutions."""
    submission_path = Path(submission_file)
    if not submission_path.exists():
        print(f"Error: Could not find submission file: {submission_path}")
        return

    mode_normalized = "compare" if mode == "!" else mode
    if mode_normalized not in ("compare", "submission"):
        print(f"Error: Unknown visualization mode '{mode}'.")
        return

    if mode_normalized == "compare":
        if solutions_file is None:
            print("Error: Solutions file required for compare mode.")
            return
        solutions_path = Path(solutions_file)
        if not solutions_path.exists():
            print(
                f"Error: Could not find solutions file for compare mode:\n{solutions_path}"
            )
            return

        with submission_path.open("r") as handle:
            subs = json.load(handle)
        with solutions_path.open("r") as handle:
            sols = json.load(handle)

        print(f"Visualizing comparison for {len(subs)} tasks...")

        for task_id, attempts_list in subs.items():
            if task_id not in sols:
                print(f"Warning: Task {task_id} not found in solutions.json")
                continue

            gt_grids = sols[task_id]
            print(gt_grids)
            for i, attempts in enumerate(attempts_list):
                if i >= len(gt_grids):
                    break

                gt = gt_grids[i]
                att1 = attempts.get("attempt_1")
                att2 = attempts.get("attempt_2")

                pass1 = (att1 == gt) if att1 is not None else False
                pass2 = (att2 == gt) if att2 is not None else False

                if pass1 and pass2:
                    status = "Pass - both"
                elif pass1:
                    status = "Pass - 1"
                elif pass2:
                    status = "Pass - 2"
                else:
                    status = "Fail"

                grids_to_plot = [gt]
                if att1 is not None:
                    grids_to_plot.append(att1)
                if att2 is not None:
                    grids_to_plot.append(att2)

                header = f"Task: {task_id} | Pair: {i} | Status: {status}"
                print(f"Plotting {header}")

                try:
                    show_grids(grids_to_plot, title=header)
                except Exception as exc:
                    print(f"Skipping plot for {task_id} due to error: {exc}")
    else:
        with submission_path.open("r") as handle:
            subs = json.load(handle)

        print(f"Visualizing submissions for {len(subs)} tasks (no solutions)...")

        for task_id, attempts_list in subs.items():
            for i, attempts in enumerate(attempts_list):
                att1 = attempts.get("attempt_1")
                att2 = attempts.get("attempt_2")

                grids_to_plot = []
                if att1 is not None:
                    grids_to_plot.append(att1)
                if att2 is not None:
                    grids_to_plot.append(att2)

                if not grids_to_plot:
                    print(f"Skipping {task_id} pair {i} (no attempts)")
                    continue

                header = f"Task: {task_id} | Pair: {i} | Status: submission-only"
                print(f"Plotting {header}")

                try:
                    show_grids(grids_to_plot, title=header)
                except Exception as exc:
                    print(f"Skipping plot for {task_id} due to error: {exc}")

def score_submission(
    solutions_file: Path, submission_file: Path
) -> Dict[str, object]:
    """Score a submission.json against ARC solutions.json."""
    solutions_path = Path(solutions_file)
    submission_path = Path(submission_file)

    if not solutions_path.exists():
        raise FileNotFoundError(f"Solutions file not found: {solutions_path}")
    if not submission_path.exists():
        raise FileNotFoundError(f"Submission file not found: {submission_path}")

    with solutions_path.open("r") as handle:
        solutions = json.load(handle)

    with submission_path.open("r") as handle:
        submissions = json.load(handle)

    calc_score = 0.0
    max_total_score = len(solutions)
    fully_solved_tasks: List[str] = []

    for task_id, ground_truth_grids in solutions.items():
        if task_id not in submissions:
            continue

        task_attempts = submissions[task_id]
        num_pairs = len(ground_truth_grids)
        pairs_solved = 0

        for i in range(min(len(task_attempts), num_pairs)):
            truth = ground_truth_grids[i]
            attempts = task_attempts[i]
            if attempts.get("attempt_1") == truth or attempts.get("attempt_2") == truth:
                pairs_solved += 1

        if num_pairs > 0:
            calc_score += pairs_solved / num_pairs
            if pairs_solved == num_pairs:
                fully_solved_tasks.append(task_id)

    percentage = 100 * (calc_score / max_total_score) if max_total_score > 0 else 0.0
    print(f"Official ARC style scoring: {calc_score}/{max_total_score} ({percentage}%)")
    print(f"Fully correct tasks ({len(fully_solved_tasks)}):")
    for task_id in fully_solved_tasks:
        print(task_id)

    return {
        "score": calc_score,
        "max_score": max_total_score,
        "percentage": percentage,
        "fully_solved_tasks": fully_solved_tasks,
    }

def free_memory(
    globals_dict: Optional[Dict[str, object]] = None,
    names: Sequence[str] = ("model", "dataset", "dataloader", "optimizer", "scheduler"),
) -> None:
    """Free common training objects and clear torch caches for inference."""
    if globals_dict is not None:
        for name in names:
            if name in globals_dict:
                del globals_dict[name]

    if hasattr(torch, "_dynamo"):
        torch._dynamo.reset()

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"GPU cleaned. Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# These allow existing code to import from utils without changes during transition

# helpers defined earlier (Cell 3)


## Training Configuration


In [9]:
args_dict = {
    "name": "arc_run",
    "data_path": DATA_DIR / "challenges.json",
    "train_log_file": CKPT_DIR / "training_log.txt",
    "save_path": CKPT_DIR / "final.pt",
    "best_save_path": CKPT_DIR / "best.pt",
    "checkpoint_path": None,
    "checkpoint_epochs": None,

    # Training
    "epochs": 50,
    "batch_size": 32,
    "gradient_accumulation_steps": 1,
    "do_validate": True,
    "val_batch_size": 70,

    # Augmentation
    "enable_aug": True,
    "max_augments": 300,
    "enable_color_aug": True,
    "color_apply_to_test": True,
    "enable_dihedral_aug": True,
    "dihedral_apply_to_test": True,

    # Optimizer (NorMuon)
    "optimizer": "normuon",
    "normuon_lr": 1.66e-3,
    "normuon_momentum": 0.95,
    "normuon_beta2": 0.95,
    "adamw_lr": 3e-4,

    # LR schedule (WSD)
    "warmup_pct": 0.02,
    "wsd_decay_start_pct": 0.8,
    "lr_floor": 0.0,

    # Regularisation
    "weight_decay": 0.1,
    "attention_weight_decay": 0.01,
    "token_embedding_weight_decay": 0.01,
    "task_embedding_weight_decay": 0.01,
    "grad_clip": 1.0,
    "dropout": 0.1,
    "attention_dropout": None,
    "seed": 42,

    # Architecture (< 50M params)
    "d_model": 512,
    "n_heads": 8,
    "d_ff": 2048,
    "n_layers": 11,

    # Inference
    "inference_temperature": None,
    "inference_top_k": None,

    # Logging
    "train_log_mode": "never",
    "log_location": "none",
}

cfg = argparse.Namespace(**args_dict)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

fix_seed(cfg.seed)
device = get_device("cuda")

_tmp_dataset = PuzzleDataset(
    json_path=cfg.data_path,
    splits=("train",),
    include_outputs=True,
    max_seq_len=MAX_LEN,
    drop_long_sequences=True,
)
_num_examples = _tmp_dataset.num_examples

_tmp_config = ARCTransformerConfig(
    num_examples=_num_examples,
    d_model=cfg.d_model,
    n_heads=cfg.n_heads,
    d_ff=cfg.d_ff,
    n_layers=cfg.n_layers,
    dropout=cfg.dropout,
    attention_dropout=cfg.attention_dropout,
)
_tmp_model = ARCTransformer(_tmp_config)

total_params = sum(p.numel() for p in _tmp_model.parameters())
trainable_params = sum(p.numel() for p in _tmp_model.parameters() if p.requires_grad)

print("=" * 60)
print("PARAMETER COUNT (Assignment Requirement)")
print("=" * 60)
print(f"  Dataset task count:   {_num_examples:>15,}")
print(f"  Total parameters:     {total_params:>15,}")
print(f"  Trainable parameters: {trainable_params:>15,}")
print(f"  Assignment limit:     {50_000_000:>15,}")
print(f"  Within budget:        {total_params <= 50_000_000}")
print("=" * 60)

if total_params > 50_000_000:
    raise ValueError(
        f"CONSTRAINT VIOLATED: Model has {total_params:,} params "
        f"(> 50,000,000). Reduce architecture size."
    )

print("\nParameter breakdown:")
for name, module in _tmp_model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:30s}: {n:>12,}  ({100*n/total_params:.1f}%)")

del _tmp_dataset, _tmp_model
torch.cuda.empty_cache()
gc.collect()

print("\nConfiguration ready.")
print(f"  Final checkpoint: {cfg.save_path}")
print(f"  Best checkpoint : {cfg.best_save_path}")
print("Run Cell 10 to start training.")

Precomputing 3D positions...
PARAMETER COUNT (Assignment Requirement)
  Dataset task count:               800
  Total parameters:          46,627,840
  Trainable parameters:      46,627,840
  Assignment limit:          50,000,000
  Within budget:        True

Parameter breakdown:
  token_embedding               :        7,168  (0.0%)
  example_embedding             :      409,600  (0.9%)
  dihedral_embedding            :        4,096  (0.0%)
  dropout                       :            0  (0.0%)
  blocks                        :   46,199,296  (99.1%)
  norm                          :          512  (0.0%)
  lm_head                       :        7,168  (0.0%)

Configuration ready.
  Final checkpoint: /content/drive/MyDrive/DL_A2/saved_models/final.pt
  Best checkpoint : /content/drive/MyDrive/DL_A2/saved_models/best.pt
Run Cell 10 to start training.


## Training Wrapper


In [10]:
@torch.no_grad()
def _compute_train_loss_for_reporting(model, dataloader, device):
    return val_epoch(model=model, dataloader=dataloader, device=device)

def run_training(
    args,
    model,
    dataloader,
    dataset,
    device,
    data_path,
    checkpoint=None,
    val_batch_size=None,
):
    """Train and save only best (lowest val loss) and final saved_models."""
    if checkpoint is None:
        checkpoint = getattr(model, "_loaded_checkpoint", None)

    optimizer = _build_optimizer(
        args,
        model,
        device,
        getattr(args, "attention_weight_decay", args.weight_decay),
        getattr(args, "token_embedding_weight_decay", 0.0),
        getattr(args, "task_embedding_weight_decay", 0.0),
    )

    steps_per_epoch = max(1, len(dataloader))
    warmup_pct = max(0.0, min(1.0, float(getattr(args, "warmup_pct", 0.02))))
    decay_start_pct = max(0.0, min(1.0, float(getattr(args, "wsd_decay_start_pct", 0.8))))
    lr_floor = max(0.0, min(1.0, float(getattr(args, "lr_floor", 0.0))))
    total_epochs = float(args.epochs)
    warmup_epochs = total_epochs * warmup_pct
    decay_start_epoch = max(warmup_epochs, total_epochs * decay_start_pct)
    decay_epochs = max(1e-8, total_epochs - decay_start_epoch)

    def lr_lambda(cur_epoch):
        if warmup_epochs > 0 and cur_epoch < warmup_epochs:
            return float(cur_epoch) / float(warmup_epochs)
        if cur_epoch < decay_start_epoch:
            return 1.0
        progress = max(0.0, min(1.0, (float(cur_epoch) - decay_start_epoch) / decay_epochs))
        return lr_floor + (1.0 - lr_floor) * (1.0 - progress)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    start_epoch = 0
    step = 0
    if checkpoint:
        step = int(checkpoint.get("global_step", 0))
        saved_epoch = checkpoint.get("epoch", checkpoint.get("epochs_completed", 0))
        start_epoch = int(saved_epoch or 0)
        if "optimizer_state" in checkpoint:
            _load_optimizer_state(optimizer, checkpoint["optimizer_state"], device)
        if "scheduler_state" in checkpoint:
            try:
                scheduler.load_state_dict(checkpoint["scheduler_state"])
            except Exception:
                pass

    val_dataloader = None
    if getattr(args, "do_validate", True):
        vb = int(val_batch_size or getattr(args, "val_batch_size", args.batch_size))
        val_dataset = PuzzleDataset(
            json_path=data_path,
            splits=("test",),
            include_outputs=True,
            load_test_solutions=True,
            max_seq_len=MAX_LEN,
            drop_long_sequences=True,
            task_whitelist=dataset.task_ids,
        )
        val_dataloader = make_dataloader(
            dataset=val_dataset,
            batch_size=vb,
            shuffle=False,
        )
        print(f"Validation dataset ready: {len(val_dataset)} examples")

    best_val_loss = float("inf")
    best_save_path = Path(getattr(args, "best_save_path", "best.pt"))
    final_save_path = Path(getattr(args, "save_path", "final.pt"))

    augmentor = getattr(dataloader, "augmentor", None)
    for epoch in range(start_epoch, int(args.epochs)):
        if augmentor is not None:
            augmentor.set_epoch(epoch)

        t0 = perf_counter()
        step = train_epoch(
            model=model,
            dataloader=dataloader,
            optimizer=optimizer,
            device=device,
            grad_clip=args.grad_clip,
            gradient_accumulation_steps=getattr(args, "gradient_accumulation_steps", 1),
            start_step=step,
            scheduler=scheduler,
            epoch=epoch,
            steps_per_epoch=steps_per_epoch,
            train_log_mode=str(getattr(args, "train_log_mode", "never")),
            log_location=str(getattr(args, "log_location", "none")),
            log_handle=None,
        )
        epoch_time = perf_counter() - t0

        train_loss = _compute_train_loss_for_reporting(model, dataloader, device)
        current_lr = scheduler.get_last_lr()[0] if scheduler is not None else optimizer.param_groups[0]["lr"]

        is_new_best = False
        if val_dataloader is not None:
            val_loss = val_epoch(model=model, dataloader=val_dataloader, device=device)
            is_new_best = val_loss < best_val_loss
            if is_new_best:
                best_val_loss = val_loss
                save_checkpoint(
                    model, dataset, data_path, best_save_path,
                    optimizer=optimizer, global_step=step, epoch=epoch + 1,
                    rng_state=save_rng(device), scheduler=scheduler,
                )
            print(
                f"Epoch {epoch + 1}/{args.epochs} | lr={current_lr:.2e} | "
                f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
                f"best_val={best_val_loss:.4f} | best_saved={is_new_best} | time={epoch_time:.1f}s"
            )
        else:
            print(
                f"Epoch {epoch + 1}/{args.epochs} | lr={current_lr:.2e} | "
                f"train_loss={train_loss:.4f} | time={epoch_time:.1f}s"
            )

    save_checkpoint(
        model, dataset, data_path, final_save_path,
        optimizer=optimizer, global_step=step, epoch=int(args.epochs),
        rng_state=save_rng(device), scheduler=scheduler,
    )
    print(f"Final checkpoint saved to: {final_save_path}")
    print(f"Best checkpoint saved to : {best_save_path}")

## Run Training


In [11]:
print("Building model and data...")
model, dataset, dataloader, device, data_path = setup_model_and_data(cfg)

print("Starting training...")
print(f"  Epochs: {cfg.epochs}")
print(f"  Batch size: {cfg.batch_size}")
print(f"  Max augments: {cfg.max_augments}")
print(f"  Optimizer: {cfg.optimizer}")
print(f"  Final checkpoint: {cfg.save_path}")
print(f"  Best checkpoint : {cfg.best_save_path}")

t_start = perf_counter()
run_training(
    cfg,
    model=model,
    dataloader=dataloader,
    dataset=dataset,
    device=device,
    data_path=data_path,
    checkpoint=getattr(model, "_loaded_checkpoint", None),
)
t_elapsed = perf_counter() - t_start

print(f"\nTraining finished in {t_elapsed/3600:.2f}h ({t_elapsed:.0f}s)")
print(f"Final checkpoint saved to: {cfg.save_path}")
print(f"Best checkpoint saved to : {cfg.best_save_path}")

Building model and data...
Precomputing 3D positions...
Augmentation stats: 2478 sequences, avg augments=220.7, min=1, max=300
Starting training...
  Epochs: 50
  Batch size: 32
  Max augments: 300
  Optimizer: normuon
  Final checkpoint: /content/drive/MyDrive/DL_A2/saved_models/final.pt
  Best checkpoint : /content/drive/MyDrive/DL_A2/saved_models/best.pt
Skipping NorMuon kernel compile (GPU: Tesla T4).
Precomputing 3D positions...
Validation dataset ready: 265 examples
Saved checkpoint to /content/drive/MyDrive/DL_A2/saved_models/best.pt
Epoch 1/50 | lr=1.66e-03 | train_loss=3.3414 | val_loss=3.2522 | best_val=3.2522 | best_saved=True | time=125.3s
Saved checkpoint to /content/drive/MyDrive/DL_A2/saved_models/best.pt
Epoch 2/50 | lr=1.66e-03 | train_loss=1.3331 | val_loss=1.4992 | best_val=1.4992 | best_saved=True | time=131.0s
Saved checkpoint to /content/drive/MyDrive/DL_A2/saved_models/best.pt
Epoch 3/50 | lr=1.66e-03 | train_loss=1.1249 | val_loss=1.2227 | best_val=1.2227 | best

## Validation Accuracy


In [12]:
@torch.no_grad()
def compute_val_accuracy(checkpoint_path: Path, data_path: Path, val_batch_size: int = 32):
    """Load a checkpoint and compute validation output-token accuracy."""
    if not checkpoint_path.exists():
        return None, None, None

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    epoch = checkpoint.get("epoch", "?")

    val_cfg = argparse.Namespace(
        data_path=data_path,
        checkpoint_path=checkpoint_path,
        enable_aug=False,
        max_augments=0,
        enable_color_aug=False,
        color_apply_to_test=False,
        enable_dihedral_aug=False,
        dihedral_apply_to_test=False,
        batch_size=val_batch_size,
        seed=42,
        device="cuda",
    )

    model, val_dataset, val_loader, device, _ = setup_model_and_data(
        val_cfg, checkpoint=checkpoint, is_eval=True
    )
    model.eval()

    total_correct = 0
    total_tokens = 0
    total_loss = 0.0

    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        example_ids = batch["example_ids"].to(device)
        dihedral_ids = batch["dihedral_ids"].to(device)
        sep_indices = batch["sep_indices"].to(device)
        positions_3d = batch["positions_3d"].to(device)
        cu_seqlens = batch.get("cu_seqlens")
        max_seqlen = batch.get("max_seqlen")

        if cu_seqlens is not None:
            cu_seqlens = cu_seqlens.to(device=device, dtype=torch.int32)
            max_seqlen = int(max_seqlen.item()) if torch.is_tensor(max_seqlen) else int(max_seqlen)
            attention_mask_in = None
        else:
            cu_seqlens = None
            max_seqlen = None
            attention_mask_in = attention_mask

        if not any(batch["has_output"]):
            continue

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            outputs = model(
                input_ids, example_ids, dihedral_ids,
                attention_mask=attention_mask_in,
                sep_indices=sep_indices,
                compute_input_loss=False,
                positions_3d=positions_3d,
                cu_seqlens=cu_seqlens,
                max_seqlen=max_seqlen,
            )

        out_loss = outputs.get("output_loss")
        num_tokens = outputs.get("num_output_tokens")
        logits = outputs.get("logits")

        if out_loss is not None and num_tokens is not None:
            n = int(num_tokens.item())
            total_loss += float(out_loss.item()) * n
            total_tokens += n

        if logits is not None and input_ids.dim() == 2:
            shift_logits = logits[:, :-1, :]
            shift_targets = input_ids[:, 1:]
            shift_mask = (attention_mask[:, 1:] if attention_mask is not None
                          else torch.ones_like(shift_targets, dtype=torch.bool))
            preds = shift_logits.argmax(dim=-1)
            shift_len = shift_targets.size(1)
            sep_pos = sep_indices.clamp(0, shift_len).unsqueeze(1)
            pos = torch.arange(shift_len, device=device).unsqueeze(0)
            output_phase = pos >= sep_pos
            valid_output = shift_mask.bool() & output_phase
            correct = (preds == shift_targets) & valid_output
            total_correct += int(correct.sum().item())

    avg_loss = total_loss / max(1, total_tokens)
    accuracy = total_correct / max(1, total_tokens)

    del model, val_dataset, val_loader
    gc.collect()
    torch.cuda.empty_cache()

    return epoch, avg_loss, accuracy

print("Evaluating validation accuracy for best/final saved_models...")
print("=" * 70)
print(f"{'Checkpoint':<40} {'Epoch':>6} {'Val Loss':>10} {'Token Acc':>10}")
print("-" * 70)

saved_models_to_eval = []
best_ckpt = Path(cfg.best_save_path) if getattr(cfg, "best_save_path", None) else None
if best_ckpt is not None and best_ckpt.exists():
    saved_models_to_eval.append(best_ckpt)
if cfg.save_path.exists() and cfg.save_path not in saved_models_to_eval:
    saved_models_to_eval.append(cfg.save_path)

if not saved_models_to_eval:
    print("No saved_models found. Run Cell 10 first.")
else:
    for ckpt in saved_models_to_eval:
        epoch_num, val_loss, acc = compute_val_accuracy(ckpt, cfg.data_path, val_batch_size=32)
        if epoch_num is not None:
            print(f"{ckpt.name:<40} {epoch_num:>6} {val_loss:>10.4f} {acc:>9.4%}")
        else:
            print(f"{ckpt.name:<40} {'N/A':>6} {'N/A':>10} {'N/A':>10}")

print("=" * 70)

Evaluating validation accuracy for best/final saved_models...
Checkpoint                                Epoch   Val Loss  Token Acc
----------------------------------------------------------------------
Precomputing 3D positions...
best.pt                                      48     0.0888   0.0000%
Precomputing 3D positions...
final.pt                                     50     0.0822   0.0000%


## Ablation Study


In [ ]:
def make_ablation_cfg(base_cfg, **overrides):
    """Return a new config namespace with the given fields overridden."""
    d = vars(base_cfg).copy()
    d.update(overrides)
    return argparse.Namespace(**d)

# Ablation A1: 3D RoPE vs 1D sinusoidal positional encoding
ablation_A1_cfg = make_ablation_cfg(
    cfg,
    name="ablation_A1_1D_rope",
    save_path=CKPT_DIR / "ablation_A1.pt",
    epochs=90,              # shorter run for ablation comparison
    max_augments=80,
    checkpoint_epochs=[],
)
print("Ablation A1 config ready: 1D sinusoidal (set rope_mode='1d' in ARCTransformer to run)")

# Ablation A2: Per-example task embedding vs no task embedding
ablation_A2_cfg = make_ablation_cfg(
    cfg,
    name="ablation_A2_no_task_emb",
    save_path=CKPT_DIR / "ablation_A2.pt",
    epochs=90,
    max_augments=80,
    checkpoint_epochs=[],
)
print("Ablation A2 config ready: no task embedding (zero out example_embedding in ARCTransformer to run)")

# Ablation A3: NorMuon + WSD schedule vs AdamW + cosine decay
ablation_A3_cfg = make_ablation_cfg(
    cfg,
    name="ablation_A3_adamw_cosine",
    save_path=CKPT_DIR / "ablation_A3.pt",
    optimizer="adamw",      # switch optimizer
    epochs=90,
    max_augments=80,
    checkpoint_epochs=[],
)
print("Ablation A3 config ready: AdamW + cosine schedule (set optimizer='adamw' and cosine sched)")

print()
print("=" * 70)
print("ABLATION RESULTS TABLE (fill in actual accuracy after running each)")
print("=" * 70)
print(f"{'Ablation':<12} {'Our Choice':<30} {'Baseline':<25} {'Acc Δ':>8}")
print("-" * 70)
print(f"{'A1 RoPE':<12} {'3D RoPE (row,col,pair_idx)':<30} {'1D sinusoidal':<25} {'TBD':>8}")
print(f"{'A2 TaskEmb':<12} {'Learned per-task embedding':<30} {'No task embedding':<25} {'TBD':>8}")
print(f"{'A3 Optim':<12} {'NorMuon + WSD schedule':<30} {'AdamW + cosine':<25} {'TBD':>8}")
print("=" * 70)
print()
print("To run an ablation:")
print("  1. Modify ARCTransformer/train_model as described in comments above")
print("  2. Pass the ablation config: setup_model_and_data(ablation_A1_cfg)")
print("  3. Call train_model(...) and run_eval_pipeline(...) with that config")
print("  4. Compare final eval accuracy to the main model's score")


## Ablation A3: AdamW Baseline


In [ ]:
import time

print("=" * 60)
print("RUNNING ABLATION A3: AdamW Optimizer Baseline")
print("=" * 60)

ablation_A3_cfg = make_ablation_cfg(
    cfg,
    name="ablation_A3_adamw",
    optimizer="adamw",
    save_path=CKPT_DIR / "ablation_A3.pt",
    best_save_path=CKPT_DIR / "ablation_A3_best.pt",
    epochs=30,
    max_augments=80,
)

print("\nBuilding model and dataloader for AdamW...")
model_A3, dataset_A3, dataloader_A3, device, data_path = setup_model_and_data(ablation_A3_cfg)

print("\n Training Ablation A3 (AdamW)...")
start_time = time.time()
run_training(
    ablation_A3_cfg,
    model=model_A3,
    dataloader=dataloader_A3,
    dataset=dataset_A3,
    device=device,
    data_path=data_path
)
print(f"Training completed in {(time.time() - start_time)/60:.1f} minutes.")

print("\nEvaluating Ablation A3...")
_, _, sub_A3 = run_eval_pipeline(
    ablation_A3_cfg,
    run_name="ablation_A3_eval",
    max_augments=ablation_A3_cfg.max_augments,
    data_path=data_path,
    checkpoint_path=ablation_A3_cfg.best_save_path
)

if sub_A3.exists() and SOLUTIONS_FILE.exists():
    score_A3 = score_submission(SOLUTIONS_FILE, sub_A3)
    print("\n" + "=" * 60)
    print(f"ABLATION A3 FINAL ACCURACY: {score_A3['percentage']:.2f}%")
    print("=" * 60)

## Ablation A1: 1D Positional Encoding


In [ ]:
import time
import types

print("=" * 60)
print("RUNNING ABLATION A1: 1D Positional Encoding Baseline")
print("=" * 60)

ablation_A1_cfg = make_ablation_cfg(
    cfg,
    name="ablation_A1_1D_rope",
    save_path=CKPT_DIR / "ablation_A1.pt",
    best_save_path=CKPT_DIR / "ablation_A1_best.pt",
    epochs=30,
    max_augments=80,
)

print("\nBuilding model and dataloader for 1D RoPE...")
model_A1, dataset_A1, dataloader_A1, device, data_path = setup_model_and_data(ablation_A1_cfg)

print("🔧 Patching RoPE3D to zero out Y and Z spatial dimensions...")
for block in model_A1.blocks:
    rope_module = block.attention.rope
    original_apply = rope_module.apply_rotary

    def apply_1d_rotary(self, q, k, pos_xyz, orig=original_apply):
        pos_1d = pos_xyz.clone()
        pos_1d[..., 1] = 0  # Y = 0
        pos_1d[..., 2] = 0  # Z = 0
        return orig(q, k, pos_1d)

    rope_module.apply_rotary = types.MethodType(apply_1d_rotary, rope_module)

print("\nTraining Ablation A1 (1D RoPE)...")
start_time = time.time()
run_training(
    ablation_A1_cfg,
    model=model_A1,
    dataloader=dataloader_A1,
    dataset=dataset_A1,
    device=device,
    data_path=data_path
)
print(f"Training completed in {(time.time() - start_time)/60:.1f} minutes.")

print("\nEvaluating Ablation A1...")
_, _, sub_A1 = run_eval_pipeline(
    ablation_A1_cfg,
    run_name="ablation_A1_eval",
    max_augments=ablation_A1_cfg.max_augments,
    data_path=data_path,
    checkpoint_path=ablation_A1_cfg.best_save_path
)

if sub_A1.exists() and SOLUTIONS_FILE.exists():
    score_A1 = score_submission(SOLUTIONS_FILE, sub_A1)
    print("\n" + "=" * 60)
    print(f"ABLATION A1 FINAL ACCURACY: {score_A1['percentage']:.2f}%")
    print("=" * 60)

## Ablation Results Summary


## Final Evaluation


In [13]:
free_memory(globals())

# Prefer best checkpoint (lowest validation loss), else final
inference_checkpoint_path = cfg.save_path
best_ckpt = Path(cfg.best_save_path) if getattr(cfg, "best_save_path", None) else None
if best_ckpt is not None and best_ckpt.exists():
    inference_checkpoint_path = best_ckpt
    print(f"Using best checkpoint: {inference_checkpoint_path}")
else:
    print(f"Best checkpoint not found; using final: {inference_checkpoint_path}")

print(f"\nStarting evaluation from: {inference_checkpoint_path}")

eval_result = run_eval_pipeline(
    cfg,
    run_name="submission_eval",
    max_augments=cfg.max_augments,
    data_path=cfg.data_path,
    checkpoint_path=inference_checkpoint_path,
    batch_size=100,
    splits=["test"],
    task_ids=None,
)

run_name, evaluation, submission_path = eval_result
SUBMISSION_FILE = submission_path
print(f"\nEvaluation complete.")
print(f"submission.json saved to: {SUBMISSION_FILE}")

GPU cleaned. Allocated: 0.02 GB
Using best checkpoint: /content/drive/MyDrive/DL_A2/saved_models/best.pt

Starting evaluation from: /content/drive/MyDrive/DL_A2/saved_models/best.pt

STARTING EVALUATION: submission_eval (Augmented augs: 300)

Building model and dataloader...
Precomputing 3D positions...
Augmentation stats: 2816 sequences, avg augments=222.4, min=1, max=300
Bypassing model compile for decoding step...
[test] Finished batch 1 / 798
[test] Finished batch 2 / 798


/usr/local/lib/python3.12/dist-packages/torch/nn/attention/flex_attention.py:1624: UserWarning: flex_attention called without torch.compile() - this will use an unfused implementation that materializes the full scores matrix instead of generating a fused kernel.

SOLUTION: Use torch.compile(flex_attention)(...)

If you want to debug your score_mod/mask_mod, you can set:
torch.nn.attention.flex_attention._FLEX_ATTENTION_DISABLE_COMPILE_DEBUG = True

This will allow you to use print statements or breakpoints. Note: This doesn't work with the backwards pass and may produce incorrect results.
  _warn_once(


[test] Finished batch 3 / 798
[test] Finished batch 4 / 798
[test] Finished batch 5 / 798
[test] Finished batch 6 / 798
[test] Finished batch 7 / 798
[test] Finished batch 8 / 798
[test] Finished batch 9 / 798
[test] Finished batch 10 / 798
[test] Finished batch 11 / 798
[test] Finished batch 12 / 798
[test] Finished batch 13 / 798
[test] Finished batch 14 / 798
[test] Finished batch 15 / 798
[test] Finished batch 16 / 798
[test] Finished batch 17 / 798
[test] Finished batch 18 / 798
[test] Finished batch 19 / 798
[test] Finished batch 20 / 798
[test] Finished batch 21 / 798
[test] Finished batch 22 / 798
[test] Finished batch 23 / 798
[test] Finished batch 24 / 798
[test] Finished batch 25 / 798
[test] Finished batch 26 / 798
[test] Finished batch 27 / 798
[test] Finished batch 28 / 798
[test] Finished batch 29 / 798
[test] Finished batch 30 / 798
[test] Finished batch 31 / 798
[test] Finished batch 32 / 798
[test] Finished batch 33 / 798
[test] Finished batch 34 / 798
[test] Finished

## Score on Evaluation Set


In [14]:
SOLUTIONS_FILE = DATA_DIR / "solutions.json"
SUBMISSION_FILE = Path("saved_models") / "eval_runs" / "submission_eval" / "submission.json"
if SUBMISSION_FILE.exists() and SOLUTIONS_FILE.exists():
    score = score_submission(SOLUTIONS_FILE, SUBMISSION_FILE)
    print("\n" + "=" * 60)
    print("FINAL ARC-1 EVALUATION RESULTS")
    print("=" * 60)
    print(f"  Score:      {score['score']:.2f} / {score['max_score']}")
    print(f"  Percentage: {score['percentage']:.2f}%")
    print(f"  Fully solved tasks: {len(score['fully_solved_tasks'])}")
    print("=" * 60)
else:
    if not SUBMISSION_FILE.exists():
        print(f"Submission file not found: {SUBMISSION_FILE}. Run Cell 12 first.")
    if not SOLUTIONS_FILE.exists():
        print(f"Solutions file not found: {SOLUTIONS_FILE}.")


Official ARC style scoring: 51.5/400 (12.875%)
Fully correct tasks (50):
137f0df0
15696249
195ba7dc
19bb5feb
1a2e2828
1c0d0a4b
21f83797
29700607
310f3251
3194b014
332efdb3
358ba94e
45737921
45bbe264
4cd1b7b2
4e469f39
506d28a5
50a16a69
54db823b
5783df64
5d2a5c43
66e6c45b
68b67ca3
705a3229
712bf12e
817e6c09
8597cfd7
8fbca751
917bccba
94414823
9c1e755f
a406ac07
a59b95c0
aa18de87
aa300dc3
af24b4cc
be03b35f
c48954c1
c7d4e6ad
ca8de6ea
cf133acc
d19f7514
d37a1ef5
da2b0fe3
e133d23d
e345f17b
e633a9e5
e69241bd
e7639916
fc754716

FINAL ARC-1 EVALUATION RESULTS
  Score:      51.50 / 400
  Percentage: 12.88%
  Fully solved tasks: 50
